# Core SWE/DSA Tracker 1: Data Structures

Split from `core_swe_dsa_tracker.ipynb`. Use this notebook as your focused DSA practice tracker.

How to use:
- Keep the checklist updated by changing `[ ]` to `[x]`.
- Under each topic, write your own code in the practice cells.
- For every problem, write brute force approach, optimized approach, complexity, and edge cases.


# Table of Contents

- [1. Data Structures](#1-data-structures)
  - [Arrays and Strings](#arrays-and-strings)
  - [Hash Tables / Dictionaries / Sets](#hash-tables--dictionaries--sets)
  - [Stacks and Queues](#stacks-and-queues)
  - [Linked Lists](#linked-lists)
  - [Trees](#trees)
  - [Binary Search Trees](#binary-search-trees)
  - [Heaps / Priority Queues](#heaps--priority-queues)
  - [Graphs](#graphs)
  - [Advanced Graph + DP](#advanced-graph--dp)
  - [DP on Graphs and Trees](#dp-on-graphs-and-trees)
  - [Tries](#tries)
  - [Disjoint Set Union / Union Find](#disjoint-set-union--union-find)


## Overall Progress Checklist

### Data Structures
- [ ] Arrays and Strings
- [ ] Hash Tables / Dictionaries / Sets
- [ ] Stacks and Queues
- [ ] Linked Lists
- [ ] Trees
- [ ] Binary Search Trees
- [ ] Heaps / Priority Queues
- [ ] Graphs
- [ ] Advanced Graph + DP
- [ ] DP on Graphs and Trees
- [ ] Tries
- [ ] Disjoint Set Union / Union Find


# 1. Data Structures

## Arrays and Strings

### Study checklist
- [ ] Array traversal, indexing, slicing, and in-place updates
- [ ] Prefix sums and difference arrays
- [ ] Two-pointer patterns on arrays and strings
- [ ] Sliding window on arrays and strings
- [ ] String frequency counting and character mapping
- [ ] Common problems: Two Sum, Best Time to Buy/Sell Stock, Product Except Self, Valid Anagram, Longest Substring Without Repeating Characters

### Notes
Write your understanding, patterns, mistakes, and edge cases here.


### Beginner-friendly intro
Arrays are ordered lists of items stored one after another in memory, and strings are just arrays of characters.
You use them when you care about positions (index 0, index 1, etc.) and want fast reads and writes by index.
Most interview problems here are about scanning, slicing, and combining values while keeping track of indices and ranges.


### Checklist item: Array traversal, indexing, slicing, and in-place updates

**Approach:**
- Why this matters: Every higher-level array technique (two pointers, sliding window, prefix sums) assumes you can index and mutate an array without off-by-one errors; get this wrong and every pattern built on top of it inherits the bug.
- Why this is the optimal approach: A single pass with `enumerate` touches each element exactly once, giving O(n) time and O(1) extra space; mutating by index instead of building a new list avoids an unnecessary O(n) allocation, which is provably optimal since any correct traversal must read all n elements at least once.
- Recognize the pattern: Decide whether you only need to read values or also mutate them; if you mutate, loop with indexes so each write is intentional, and only mutate the input in place when the problem doesn't require preserving the original.
- Code walkthrough:
- Uses `enumerate` to access both index and value in one loop over `nums = [3, 1, 4, 1, 5]`.
- Accumulates `running_total` from the original values before any mutation.
- Doubles odd values in-place with `nums[index] = value * 2`, showing safe index-based mutation without creating a new list.

**Learn more:**
- Website: [GeeksforGeeks: Array Data Structure](https://www.geeksforgeeks.org/array-data-structure-guide/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Array+traversal+data+structures+algorithms)

**Trade-offs:**
- In-place mutation saves memory but destroys the original array; if a caller elsewhere still needs `nums` unmodified, this traversal silently corrupts their data.
- Doubling odd values in-place is O(1) extra space; the alternative of building a new list with a comprehension is more idiomatic Python but costs an extra O(n) allocation for no benefit here.

**Practical software engineering use cases:**
- When to use it: Use direct index loops when you need to both read and conditionally mutate elements in one pass, such as normalizing values or flagging rows during ETL.
- When not to use it: Do not use manual index loops for pure transformations; `[f(x) for x in nums]` or vectorized NumPy/pandas operations are clearer and often faster than hand-rolled index mutation.


In [1]:
# Array traversal, indexing, slicing, and in-place updates
nums = [3, 1, 4, 1, 5]
running_total = 0
for index, value in enumerate(nums):
    running_total += value
    if value % 2 == 1:
        nums[index] = value * 2

print(nums)
print(running_total)


[6, 2, 4, 2, 10]
14


### Checklist item: Prefix sums and difference arrays

**Approach:**
- Why this matters: Repeated range-sum queries over the same array are a common production hot path (dashboards, analytics rollups); recomputing sum(nums[left:right+1]) per query is the naive approach that silently becomes a bottleneck as query volume grows.
- Why this is the optimal approach: Building one cumulative-sum array costs O(n) once; after that every range query is a single subtraction, O(1). This is optimal because any algorithm must read all n elements at least once to account for their values (an Omega(n) lower bound), so O(n) preprocessing plus O(1) query is asymptotically the best possible.
- Recognize the pattern: Precompute cumulative state so each range query or batch update avoids rescanning the original array; use a sentinel 0 at prefix[0] so range_sum(left, right) is a clean subtraction with no off-by-one special case.
- Code walkthrough:
- Builds `prefix` starting with a sentinel `0` so every range maps cleanly: `prefix[i+1]` holds the sum of `nums[:i+1]`.
- Defines `range_sum(left, right)` as a single subtraction: `prefix[right + 1] - prefix[left]`, giving O(1) per query after O(n) setup.
- Calls `range_sum(1, 3)` to return the sum of indices 1–3 (values 4+1+7 = 12).

**Learn more:**
- Website: [GeeksforGeeks: Array Data Structure](https://www.geeksforgeeks.org/array-data-structure-guide/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Prefix+sums+and+difference+arrays+data+structures+algorithms)

**Trade-offs:**
- Compared to recomputing each range sum directly, the prefix-sum array trades O(n) upfront memory and setup time for O(1) queries — a clear win when queries outnumber array rebuilds.
- Prefix sums assume the array is static between queries; if elements are updated frequently, rebuilding the whole prefix array per update costs O(n), and a Fenwick tree / BIT (O(log n) update and query) becomes the better trade-off.

**Practical software engineering use cases:**
- When to use it: Use this when the array is read-heavy with many range-sum queries and few or no updates, e.g., serving cumulative revenue-by-date-range from a fixed daily-totals array.
- When not to use it: Do not use a plain prefix-sum array when updates are frequent and interleaved with queries; use a Fenwick tree or segment tree instead so updates don't cost O(n) each.


In [2]:
# Prefix sums answer repeated range-sum queries in O(1) after O(n) setup.
nums = [2, 4, 1, 7, 3]
prefix = [0]
for value in nums:
    prefix.append(prefix[-1] + value)

def range_sum(left, right):
    return prefix[right + 1] - prefix[left]

print(range_sum(1, 3))


12


### Checklist item: Two-pointer patterns on arrays and strings

**Approach:**
- Why this matters: Two pointers replace the naive nested-loop scan (checking every pair) that many array problems tempt you into, which matters because that nested loop is the single most common source of accidental O(n^2) code in interviews and production alike.
- Why this is the optimal approach: On a sorted array, moving `left` right when the sum is too small and `right` left when it's too large is optimal because at each step you eliminate an entire row or column of the O(n^2) pair-search space: if nums[left]+nums[right] < target, no pair using the current right and any index >= left works, so advancing left is safe. This gives O(n) time versus O(n^2) brute force, and each pointer moves at most n times total, so total work is bounded by 2n.
- Recognize the pattern: Decide what each pointer represents, write the condition for moving each pointer, then prove no valid answer is skipped — for a sorted array, monotonicity guarantees that moving the pointer on the 'wrong side' of target never eliminates a real solution.
- Code walkthrough:
- Places `left` at index 0 and `right` at the last index, then moves them inward based on whether the current sum is too small or too large.
- Advances `left` when the sum is below `target`; retreats `right` when above — guaranteed to find the pair in a sorted array without revisiting any index.
- Prints the index pair `(1, 3)` when `nums[1] + nums[3] == 9`, then breaks immediately.

**Learn more:**
- Website: [GeeksforGeeks: Array Data Structure](https://www.geeksforgeeks.org/array-data-structure-guide/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Two-pointer+patterns+on+arrays+and+strings+data+structures+algorithms)

**Trade-offs:**
- Two pointers require the array to be sorted (or sortable); if the input arrives unsorted and you can't sort it (e.g., you must preserve original indices), you need a hash-map approach instead, trading O(1) space for O(n) space.
- Compared to a hash-map one-pass approach (also O(n) time but O(n) space), two pointers use O(1) extra space but require a sort step (O(n log n)) if the array isn't already sorted, so the right choice depends on whether sortedness is free or must be paid for.

**Practical software engineering use cases:**
- When to use it: Use this on sorted arrays/strings for pair-sum, palindrome checks, container/area problems, or merging, where you can reason about monotonic pointer movement.
- When not to use it: Do not force two pointers on unsorted data where sorting would destroy needed information (like original indices); use a hash map instead.


In [3]:
# Two pointers on a sorted array.
nums = [1, 2, 4, 7, 11]
target = 9
left, right = 0, len(nums) - 1
while left < right:
    total = nums[left] + nums[right]
    if total == target:
        print((left, right))
        break
    if total < target:
        left += 1
    else:
        right -= 1


(1, 3)


### Checklist item: Sliding window on arrays and strings

**Approach:**
- Why this matters: Sliding window replaces the brute-force approach of recomputing a window's sum or state from scratch at every position, which matters because that recomputation turns an O(n) scan into an O(n*k) or O(n^2) one for problems with many overlapping windows.
- Why this is the optimal approach: Maintaining a running window and updating it incrementally (add the incoming element, remove the outgoing one) does O(1) work per step instead of O(k) work to recompute a fresh window sum, giving O(n) total time — optimal because you cannot compute an answer over every position without touching each element at least once, and this approach touches each element a constant number of times (once in, once out).
- Recognize the pattern: Define the valid-window condition, expand with the right pointer, update state incrementally, then shrink from the left only when the condition is violated — never recompute the window from scratch.
- Code walkthrough:
- Computes the initial window sum for the first `k = 3` elements with `sum(nums[:k])` and stores it in `best`.
- Slides one step at a time: adds the incoming right element (`nums[right]`) and subtracts the outgoing left element (`nums[right - k]`).
- Keeps `best` as the running maximum, so only one full scan is needed after initialisation.

**Learn more:**
- Website: [GeeksforGeeks: Array Data Structure](https://www.geeksforgeeks.org/array-data-structure-guide/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Sliding+window+on+arrays+and+strings+data+structures+algorithms)

**Trade-offs:**
- Fixed-size sliding window (as here) is simpler than variable-size sliding window, but only fits problems with a known window length; variable-size windows need an explicit shrink-while-invalid loop instead of a fixed step.
- The incremental update (subtract outgoing, add incoming) is O(1) per step, but it only works for associative, invertible aggregates like sum; for aggregates like max/min you need a monotonic deque to maintain O(1) amortized updates instead.

**Practical software engineering use cases:**
- When to use it: Use this for contiguous-subarray/substring problems with a fixed or growing/shrinking window: max sum of k elements, longest substring under a constraint, rate-limiting over a time window.
- When not to use it: Do not use sliding window when the aggregate you need (like max) isn't cheaply updatable by simple add/remove; reach for a monotonic deque or heap-backed window instead.


In [4]:
# Sliding window: max sum of any k consecutive values.
nums = [4, 2, 1, 7, 8, 1]
k = 3
window = sum(nums[:k])
best = window
for right in range(k, len(nums)):
    window += nums[right] - nums[right - k]
    best = max(best, window)
print(best)


16


### Checklist item: String frequency counting and character mapping

**Approach:**
- Why this matters: Character/word frequency counting underlies anagram checks, first-unique-character problems, and text analytics; doing this with nested loops (checking each character against all others) is the naive trap that costs O(n^2) instead of O(n).
- Why this is the optimal approach: A single pass using `dict.get(key, 0) + 1` builds the full frequency table in O(n) time and O(k) space (k = number of distinct characters), which is optimal because you must inspect every character at least once to know its count, and a hash map gives O(1) amortized read/write per character.
- Recognize the pattern: Choose the key you need for lookup or grouping, update the dictionary as you scan, and use that accumulated state to avoid nested loops or repeated scans over the same data.
- Code walkthrough:
- Iterates over every character in `'interview'` using `counts.get(ch, 0)` to safely fetch the existing count or default to zero, then writes `count + 1`.
- Produces a dict where each key is a character and each value is how many times it appears — built in a single O(n) pass.

**Learn more:**
- Website: [GeeksforGeeks: Array Data Structure](https://www.geeksforgeeks.org/array-data-structure-guide/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=String+frequency+counting+and+character+mapping+data+structures+algorithms)

**Trade-offs:**
- A plain dict works well for a small, unknown alphabet, but for a fixed small alphabet (like lowercase a-z) a 26-length array indexed by `ord(ch) - ord('a')` is faster and uses less memory than a hash map.
- `counts.get(ch, 0) + 1` is simple and readable; `collections.Counter(text)` does the identical O(n) counting in one call and additionally offers `.most_common()`, so prefer it in production code once you understand the manual version.

**Practical software engineering use cases:**
- When to use it: Use this for anagram/permutation checks, character-frequency validation, and any text feature that reduces to 'how many times does X appear'.
- When not to use it: Do not hand-roll frequency counting in production; use `collections.Counter`, which is the same O(n) algorithm with a clearer, tested interface.


In [5]:
# Frequency counting with a dictionary.
text = "interview"
counts = {}
for ch in text:
    counts[ch] = counts.get(ch, 0) + 1
print(counts)


{'i': 2, 'n': 1, 't': 1, 'e': 2, 'r': 1, 'v': 1, 'w': 1}


### Checklist item: Common problems: Two Sum, Best Time to Buy/Sell Stock, Product Except Self, Valid Anagram, Longest Substring Without Repeating Characters

**Approach:**
- Why this matters: A checklist of named problems is only useful if you actually implement one end-to-end; the item exists to force the jump from recognizing a pattern to writing a working, tested solution under the section's core technique (hash maps for O(1) lookups instead of nested loops).
- Why this is the optimal approach: Two Sum's optimal solution is O(n) time and O(n) space: for each number, check whether its complement (target - num) was already seen, using a hash map for O(1) average lookup. This is optimal because any algorithm must read all n numbers at least once (Omega(n)), and the hash-map pass achieves that lower bound; the naive nested-loop check of every pair is O(n^2) and does strictly more work than necessary.
- Recognize the pattern: Use the list as a practice queue: pick one problem, write the brute-force version first, identify the repeated work (re-scanning for a complement), then replace that repeated work with a hash map lookup.
- Code walkthrough:
- Defines `two_sum(nums, target)`: scans once, and for each `num` checks whether `target - num` is already a key in `seen` before inserting `num` itself.
- Because the complement check happens before the insert, a value can't pair with itself unless it appears twice in the input.
- On `[2, 7, 11, 15]` with target `9`, `7`'s complement `2` is already in `seen` from index 0, so it returns `[0, 1]` in a single O(n) pass.

**Learn more:**
- Website: [GeeksforGeeks: Array Data Structure](https://www.geeksforgeeks.org/array-data-structure-guide/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Two+Sum+data+structures+algorithms)

**Trade-offs:**
- The hash-map approach uses O(n) extra space to buy O(n) time; if memory is extremely constrained and the array is sorted (or can be sorted), the two-pointer technique from earlier in this section solves the same problem in O(1) extra space.
- This solution assumes exactly one valid answer exists and returns immediately on the first match; a version that must find all pairs summing to target needs to keep scanning and collect multiple results instead of returning early.

**Practical software engineering use cases:**
- When to use it: Use a hash-map lookup whenever you need to find a complement, pair, or previously-seen value in O(1) rather than rescanning the array.
- When not to use it: Don't reach for a hash map if the array is already sorted and you only need one pass with O(1) space — two pointers solves it just as fast with less memory.


In [ ]:
# Arrays and Strings - Common problems: Two Sum, Best Time to Buy/Sell Stock, Product Except Self, Valid Anagram, Longest Substring Without Repeating Characters
# Two Sum solved in full below; the remaining problems stay on the practice queue.
def two_sum(nums, target):
    seen = {}
    for i, num in enumerate(nums):
        complement = target - num
        if complement in seen:
            return [seen[complement], i]
        seen[num] = i
    return []

print(two_sum([2, 7, 11, 15], 9))  # [0, 1] because nums[0] + nums[1] == 9

practice_queue = [
    {'problem': 'Best Time to Buy/Sell Stock', 'topic': 'Arrays and Strings', 'status': 'todo'},
    {'problem': 'Product Except Self', 'topic': 'Arrays and Strings', 'status': 'todo'},
    {'problem': 'Valid Anagram', 'topic': 'Arrays and Strings', 'status': 'todo'},
    {'problem': 'Longest Substring Without Repeating Characters', 'topic': 'Arrays and Strings', 'status': 'todo'}
]

for entry in practice_queue:
    print(f"{entry['topic']}: {entry['problem']} -> {entry['status']}")


In [7]:
# Practice: Arrays and Strings

# Problem:
# Approach:
# Time Complexity:
# Space Complexity:
# Edge Cases:



## Hash Tables / Dictionaries / Sets

### Study checklist
- [ ] Hash map insert, lookup, update, and delete
- [ ] Frequency maps
- [ ] Duplicate detection
- [ ] Set-based lookup
- [ ] Grouping problems such as anagrams
- [ ] Common problems: Contains Duplicate, Group Anagrams, Top K Frequent Elements, Longest Consecutive Sequence

### Notes
Write your understanding, patterns, mistakes, and edge cases here.


### Beginner-friendly intro
Hash tables (maps/dictionaries) store key–value pairs and give you very fast average lookups by key.
Sets are like hash tables that only care whether an element exists, not about any attached value.
You use them when you need to check membership, count frequencies, or group items by some key efficiently.


### Checklist item: Hash map insert, lookup, update, and delete

**Approach:**
- Why this matters: CRUD operations on a hash map are the foundation every other item in this section builds on; if you don't know that `.get` avoids a KeyError and `del` raises one on a missing key, every 'clever' hash-map trick later becomes a landmine.
- Why this is the optimal approach: Hashing the key to a bucket index gives O(1) average-case insert, lookup, update, and delete, because the hash function computes the storage location directly instead of searching — this is optimal versus a list-based key/value store, which needs O(n) linear search for the same operations.
- Recognize the pattern: Choose the key you need for lookup, mutate the dictionary directly for insert/update, and always use `.get(key, default)` for reads that might miss so a missing key doesn't crash the program with a KeyError.
- Code walkthrough:
- Inserts `'alice': 10` with direct assignment, then increments in-place with `+= 5`.
- Reads with `.get('alice', 0)` — the default avoids a `KeyError` when the key might be absent.
- Deletes with `del` and confirms removal with an `in` check, which returns `False`.

**Learn more:**
- Website: [GeeksforGeeks: Hashing in Data Structure](https://www.geeksforgeeks.org/hashing-data-structure/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Hash+map+insert+data+structures+algorithms)

**Trade-offs:**
- Average-case O(1) operations assume a good hash function and low collision rate; a pathological hash (or a maliciously crafted key set) degrades to O(n) worst case, which is why Python randomizes string hashing by default.
- Using `+=` on a dict value (`scores['alice'] += 5`) is only safe once the key already exists; on a fresh key it raises KeyError, so production code should use `scores['alice'] = scores.get('alice', 0) + 5` or a `defaultdict(int)`.

**Practical software engineering use cases:**
- When to use it: Use hash maps for caches, session stores, request de-duplication, or any lookup keyed by an ID where insert/update/delete all need to be fast.
- When not to use it: Do not use a plain dict when you need sorted iteration order or range queries by key; use a sorted structure (e.g., a balanced tree or `sortedcontainers.SortedDict`) instead.


In [8]:
# Hash map CRUD operations.
scores = {}
scores["alice"] = 10
scores["alice"] += 5
print(scores.get("alice", 0))
del scores["alice"]
print("alice" in scores)


15
False


### Checklist item: Frequency maps

**Approach:**
- Why this matters: Frequency maps are the single most reused idea in string/array interview problems (anagrams, majority element, mode); building this incrementally in one pass is the difference between an O(n) and an accidental O(n^2) solution.
- Why this is the optimal approach: Updating `counts[word] = counts.get(word, 0) + 1` for each element does O(1) amortized work per element, giving O(n) total time to build the full frequency table — optimal because you must examine every element at least once to count it.
- Recognize the pattern: Choose the key you need for grouping (here, the word itself), update the dictionary as you scan, and read that accumulated state instead of re-scanning the list for each distinct value.
- Code walkthrough:
- Scans `['go', 'go', 'stop']` once, using `counts.get(word, 0) + 1` to increment each word's tally without a separate initialisation step.
- Produces `{'go': 2, 'stop': 1}` in a single O(n) pass — no nested loops needed.

**Learn more:**
- Website: [GeeksforGeeks: Hashing in Data Structure](https://www.geeksforgeeks.org/hashing-data-structure/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Frequency+maps+data+structures+algorithms)

**Trade-offs:**
- A frequency dict costs O(k) extra memory for k distinct keys; if the alphabet of possible keys is small and fixed, a fixed-size array indexed by key is faster and uses less memory than a hash map.
- `counts.get(word, 0) + 1` handles the first-occurrence case explicitly; `collections.Counter(words)` computes the identical result in one call with less boilerplate and should be preferred in production code.

**Practical software engineering use cases:**
- When to use it: Use this whenever you need 'how many times does X occur' as an intermediate step — majority element, mode, histogram building, event counting.
- When not to use it: Do not build a frequency map when you only need to check membership (yes/no), not counts; a plain set is cheaper in memory and intent.


In [9]:
# Frequency map.
words = ["go", "go", "stop"]
counts = {}
for word in words:
    counts[word] = counts.get(word, 0) + 1
print(counts)


{'go': 2, 'stop': 1}


### Checklist item: Duplicate detection

**Approach:**
- Why this matters: Detecting duplicates is the textbook case for trading memory for speed; the naive nested-loop comparison of every pair is O(n^2) and is exactly the mistake this pattern exists to prevent.
- Why this is the optimal approach: Checking `value in seen` before adding to a set is O(1) average per check, so scanning once and testing membership gives O(n) total time versus O(n^2) for pairwise comparison — optimal because a duplicate can only be confirmed by having seen its earlier occurrence, which a set lets you check in constant time.
- Recognize the pattern: Choose the key you need for lookup (the value itself here), update the set as you scan, and use membership testing to avoid a nested loop that re-compares every pair.
- Code walkthrough:
- Checks `value in seen` before adding to the set; if already present, a duplicate is found and the loop breaks.
- Breaks on the first duplicate (`4`) rather than continuing, so the scan stops as early as possible.

**Learn more:**
- Website: [GeeksforGeeks: Hashing in Data Structure](https://www.geeksforgeeks.org/hashing-data-structure/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Duplicate+detection+data+structures+algorithms)

**Trade-offs:**
- A set-based scan is O(n) time and O(n) space; if memory is tight and the input is sortable, sorting first (O(n log n) time, O(1) extra space) and checking adjacent pairs trades time for space.
- This implementation stops at the first duplicate found; a version that needs to report all duplicates (not just detect the first) must keep scanning and collect matches instead of breaking early.

**Practical software engineering use cases:**
- When to use it: Use a set for fast duplicate/membership checks in de-duplication pipelines, form-validation, or checking whether an ID has already been processed.
- When not to use it: Do not use a set when you need to preserve or count occurrences — a set collapses duplicates, so use a list plus a frequency dict if occurrence counts matter.


In [10]:
# Detect duplicates in O(n) time with a set.
nums = [4, 1, 9, 4]
seen = set()
for value in nums:
    if value in seen:
        print("duplicate:", value)
        break
    seen.add(value)


duplicate: 4


### Checklist item: Set-based lookup

**Approach:**
- Why this matters: Membership checks inside a loop (`if x in some_list`) are an easy performance trap: a Python list's `in` operator is O(n) per check, silently turning a loop that looks like O(n) into O(n*m).
- Why this is the optimal approach: Converting the lookup collection to a set makes each `in` check O(1) average, so filtering m requests against an n-item allowlist costs O(n + m) instead of O(n*m) with a list — optimal because a hash set is the standard data structure for O(1) average membership testing.
- Recognize the pattern: Choose the key you need for lookup, build the set once outside the loop, and use `in` membership tests to avoid an inner loop entirely.
- Code walkthrough:
- Creates `allowed` as a set for O(1) membership tests, then filters `requests` in a single list comprehension.
- Only `'read'` and `'write'` satisfy `in allowed`; `'delete'` is excluded without any nested loop.

**Learn more:**
- Website: [GeeksforGeeks: Hashing in Data Structure](https://www.geeksforgeeks.org/hashing-data-structure/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Set-based+lookup+data+structures+algorithms)

**Trade-offs:**
- Building the `allowed` set costs O(n) upfront memory and time; for a very small, fixed allowlist checked only once, the conversion overhead may not be worth it, but it pays off immediately once checked more than once.
- Sets guarantee O(1) average membership but O(n) worst case under hash collisions; if keys are attacker-controlled and collision attacks are a concern, use a hash function resistant to that or a different structure.

**Practical software engineering use cases:**
- When to use it: Use a set for allowlist/denylist checks, permission lookups, or any 'is this value in this collection' test performed repeatedly inside a loop.
- When not to use it: Do not convert to a set if you need order preservation or duplicate counts; a set discards both.


In [11]:
# Set lookup avoids an inner loop.
allowed = {"read", "write"}
requests = ["read", "delete", "write"]
print([request for request in requests if request in allowed])


['read', 'write']


### Checklist item: Grouping problems such as anagrams

**Approach:**
- Why this matters: Grouping items by a derived key (like sorted characters for anagrams) is the generalization of frequency counting; without a canonical key, you're stuck comparing every pair of words to each other, which is O(n^2 * k) for n words of length k.
- Why this is the optimal approach: Computing a canonical key per word (sorting its characters, O(k log k)) and grouping by that key in a dict gives O(n * k log k) total time — optimal because any two anagrams must share the same sorted form, so this key perfectly partitions the input with a single pass instead of pairwise comparison.
- Recognize the pattern: Choose the key you need for grouping (a canonical, order-independent representation), use `setdefault` to lazily initialize each group's list, then append to it as you scan.
- Code walkthrough:
- Computes a canonical sort-key for each word: `'eat'` and `'tea'` both sort to `'aet'`, mapping them to the same group.
- Uses `setdefault` to initialise an empty list on the first occurrence of a key, then appends subsequent anagrams to that list.
- Prints the grouped values where each inner list contains all words that are anagrams of each other.

**Learn more:**
- Website: [GeeksforGeeks: Hashing in Data Structure](https://www.geeksforgeeks.org/hashing-data-structure/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Grouping+problems+such+as+anagrams+data+structures+algorithms)

**Trade-offs:**
- Sorting each word's characters to build the key costs O(k log k) per word; an alternative canonical key using a 26-length character-count tuple builds the key in O(k) instead, which is faster for long words but only works for a small fixed alphabet.
- `setdefault(key, []).append(word)` is concise but allocates an empty list even when about to append immediately after; `collections.defaultdict(list)` expresses the same intent and is the idiomatic production choice.

**Practical software engineering use cases:**
- When to use it: Use canonical-key grouping for anagram detection, deduplicating near-identical records, or bucketing items by a derived signature.
- When not to use it: Do not use sort-based keys when the alphabet is large (e.g., unicode text) and a cheaper hashable signature exists — sorting cost then dominates unnecessarily.


In [12]:
# Group anagrams by a sorted-character key.
words = ["eat", "tea", "tan", "ate", "nat", "bat"]
groups = {}
for word in words:
    key = "".join(sorted(word))
    groups.setdefault(key, []).append(word)
print(list(groups.values()))


[['eat', 'tea', 'ate'], ['tan', 'nat'], ['bat']]


### Checklist item: Common problems: Contains Duplicate, Group Anagrams, Top K Frequent Elements, Longest Consecutive Sequence

**Approach:**
- Why this matters: This item exists to turn hash-map pattern recognition into a working solution on a genuinely different, heap-plus-hash problem than the frequency-map and grouping items already covered, so the 'top-k by frequency' combination gets real practice.
- Why this is the optimal approach: Counting frequencies with a hash map is O(n); selecting the k largest counts with a heap (`heapq.nlargest`) costs O(n log k) instead of sorting all distinct counts (O(d log d) for d distinct values) — optimal when k is small relative to the number of distinct elements, since you never need a full sort, only the top k.
- Recognize the pattern: Use the list as a practice queue: pick one problem, write the brute-force version first (sort all counts descending and slice the first k), identify the repeated work (fully sorting when only the top k matter), then replace it with a heap-based selection.
- Code walkthrough:
- Defines `top_k_frequent(nums, k)`: builds a frequency table with `collections.Counter`, then uses `heapq.nlargest(k, ...)` keyed on frequency to select the k most common values in O(n log k).
- `heapq.nlargest` internally maintains a size-k heap rather than sorting every distinct value, so it's faster than `sorted(counts.items())[:k]` when the number of distinct values is much larger than k.
- On `[1, 1, 1, 2, 2, 3]` with `k=2`, returns `[1, 2]` — the two most frequent values, in frequency order.

**Learn more:**
- Website: [GeeksforGeeks: Hashing in Data Structure](https://www.geeksforgeeks.org/hashing-data-structure/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Contains+Duplicate+data+structures+algorithms)

**Trade-offs:**
- `heapq.nlargest` is convenient but internally still does O(d log k) work; for very large k close to d, a full sort (O(d log d)) is simpler and not meaningfully slower — the heap only wins clearly when k is small.
- This returns values only, discarding the exact counts; if downstream code needs the counts too, return `(value, count)` pairs instead of unwrapping them.

**Practical software engineering use cases:**
- When to use it: Use a hash map plus heap for 'top k by frequency/score' problems: trending hashtags, most-visited pages, most frequent error codes.
- When not to use it: Don't use a heap when k is close to the total number of distinct items — just sort everything; the heap's advantage disappears and sorting is simpler to read.


In [ ]:
# Hash Tables / Dictionaries / Sets - Common problems: Contains Duplicate, Group Anagrams, Top K Frequent Elements, Longest Consecutive Sequence
# Top K Frequent Elements solved in full below; the remaining problems stay on the practice queue.
import heapq
from collections import Counter

def top_k_frequent(nums, k):
    counts = Counter(nums)
    return [num for num, _ in heapq.nlargest(k, counts.items(), key=lambda pair: pair[1])]

print(top_k_frequent([1, 1, 1, 2, 2, 3], 2))  # [1, 2]

practice_queue = [
    {'problem': 'Contains Duplicate', 'topic': 'Hash Tables / Dictionaries / Sets', 'status': 'todo'},
    {'problem': 'Group Anagrams', 'topic': 'Hash Tables / Dictionaries / Sets', 'status': 'todo'},
    {'problem': 'Longest Consecutive Sequence', 'topic': 'Hash Tables / Dictionaries / Sets', 'status': 'todo'}
]

for entry in practice_queue:
    print(f"{entry['topic']}: {entry['problem']} -> {entry['status']}")


In [14]:
# Practice: Hash Tables / Dictionaries / Sets

# Problem:
# Approach:
# Time Complexity:
# Space Complexity:
# Edge Cases:



## Stacks and Queues

### Study checklist
- [ ] Stack operations and LIFO reasoning
- [ ] Queue and deque operations
- [ ] Valid parentheses
- [ ] Monotonic stack
- [ ] BFS queue usage
- [ ] Common problems: Valid Parentheses, Min Stack, Daily Temperatures, Evaluate Reverse Polish Notation

### Notes
Write your understanding, patterns, mistakes, and edge cases here.


### Beginner-friendly intro
Stacks are Last-In-First-Out: the last item you put in comes out first, like an undo history.
Queues are First-In-First-Out: the first item you put in comes out first, like people standing in a line.
You use stacks when you model nested operations or backtracking, and queues when you process items in arrival order (like BFS).


### Checklist item: Stack operations and LIFO reasoning

**Approach:**
- Why this matters: LIFO order is the entire reason a stack is useful: it lets you resolve the most recently opened thing first, which is exactly the shape of nested structures (function calls, brackets, undo history) that arrays alone don't model cleanly.
- Why this is the optimal approach: Python lists implement `append`/`pop` from the end in O(1) amortized time because no elements need to shift — using the end of a list as a stack is optimal versus using the front (`pop(0)`, `insert(0, x)`), which is O(n) because every other element must shift.
- Recognize the pattern: Define what belongs on the stack and when it becomes resolved, then pop only when the current item proves a previous item is complete — never pop speculatively.
- Code walkthrough:
- Pushes `'open'`, `'work'`, and `'close'` with `append`, so `'close'` ends up on top.
- Drains the stack with `pop`, printing `'close'`, `'work'`, `'open'` in LIFO order — last pushed is first popped.

**Learn more:**
- Website: [GeeksforGeeks: Stack Data Structure](https://www.geeksforgeeks.org/stack-data-structure/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Stack+operations+and+LIFO+reasoning+data+structures+algorithms)

**Trade-offs:**
- Using a Python list as a stack is simple and fast (O(1) push/pop at the end), but it offers no protection against popping an empty stack — always guard with `if stack:` before popping in production code.
- A stack enforces strict LIFO order; if you ever need to peek at or remove an item that isn't the most recent, a stack is the wrong structure and you need a list, deque, or priority queue instead.

**Practical software engineering use cases:**
- When to use it: Use a stack for undo/redo history, call-stack simulation, or any 'most recent thing must be handled first' workflow.
- When not to use it: Do not use a stack when you need first-in-first-out order or random access to elements in the middle — that's a queue or a list, respectively.


In [15]:
# Stack: last in, first out.
stack = []
for token in ["open", "work", "close"]:
    stack.append(token)
while stack:
    print(stack.pop())


close
work
open


### Checklist item: Queue and deque operations

**Approach:**
- Why this matters: A plain Python list used as a queue (`pop(0)`) is a classic hidden performance bug: it looks correct but is O(n) per dequeue because every remaining element must shift left, which quietly turns an O(n) processing loop into O(n^2).
- Why this is the optimal approach: `collections.deque` is implemented as a doubly linked list of blocks, giving O(1) `append` and O(1) `popleft` — optimal for FIFO processing because both ends of the queue are touched in constant time, unlike a list where the front is O(n).
- Recognize the pattern: Append to the right and remove from the left for queue semantics; always use `deque` for this instead of a list so `popleft` stays O(1) instead of degrading to O(n).
- Code walkthrough:
- Creates a deque pre-loaded with two jobs, then appends `'job3'` to the right end.
- Drains left-to-right with `popleft`, printing jobs in FIFO arrival order: `job1`, `job2`, `job3`.

**Learn more:**
- Website: [GeeksforGeeks: Stack Data Structure](https://www.geeksforgeeks.org/stack-data-structure/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Queue+and+deque+operations+data+structures+algorithms)

**Trade-offs:**
- `deque` gives O(1) operations at both ends but O(n) access to elements in the middle (no fast random indexing like a list); if you need indexed access, a list or array is the better fit.
- A plain `list.pop(0)` is more familiar-looking but is O(n) per call; the switch to `deque.popleft()` is a one-line change that fixes an O(n^2) processing loop into O(n) — always prefer `deque` for queue workloads.

**Practical software engineering use cases:**
- When to use it: Use a deque-backed queue for job queues, BFS traversal, rate limiters, or any first-in-first-out processing order.
- When not to use it: Do not use a queue when priority (not arrival order) should determine what's processed next — that calls for a heap-based priority queue instead.


In [16]:
# Queue with deque: first in, first out.
from collections import deque
queue = deque(["job1", "job2"])
queue.append("job3")
while queue:
    print(queue.popleft())


job1
job2
job3


### Checklist item: Valid parentheses

**Approach:**
- Why this matters: Balanced-bracket validation is the simplest real demonstration of why LIFO order matters: an opener must be closed by the *most recently opened, still-unclosed* bracket, which is precisely what a stack tracks and a simple counter cannot.
- Why this is the optimal approach: Pushing openers and popping-and-matching on closers does O(1) work per character, giving O(n) total time and O(n) worst-case space — optimal because you must inspect every character at least once, and the stack gives the only O(1) way to know 'what unmatched opener is currently innermost'.
- Recognize the pattern: Define what belongs on the stack and when it becomes resolved: openers go on the stack; a closer must match the top of the stack or the string is invalid, and any openers left unmatched at the end also make it invalid.
- Code walkthrough:
- Builds a `pairs` dict mapping each closer to its expected opener.
- Pushes every opener onto `stack`; on a closer, pops and checks whether the top matches — if not, returns `False` immediately.
- Returns `not stack` at the end so any unclosed opener also causes a failure.

**Learn more:**
- Website: [GeeksforGeeks: Stack Data Structure](https://www.geeksforgeeks.org/stack-data-structure/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Valid+parentheses+data+structures+algorithms)

**Trade-offs:**
- A counter (`+1` for `(`, `-1` for `)`) is O(1) space and works for a single bracket type, but fails on mixed bracket types like `{[(])}` where a counter alone can't detect that the innermost opener isn't the matching type — you need the stack's order information.
- Returning `False` immediately on a mismatch is faster than scanning to the end, but if you need to report *where* the first invalid character is (not just whether the string is valid), you must track and return the index at the point of failure.

**Practical software engineering use cases:**
- When to use it: Use stack-based matching for any nested-delimiter validation: brackets, XML/HTML tag balance, nested comments, or parser syntax checks.
- When not to use it: Do not use this pattern for non-nested delimiter checks (e.g., just counting total open vs. close characters) — a simple counter is simpler and sufficient there.


In [17]:
# Validate balanced parentheses with a stack.
def is_valid(s):
    pairs = {")": "(", "]": "[", "}": "{"}
    stack = []
    for ch in s:
        if ch in pairs.values():
            stack.append(ch)
        elif ch in pairs and (not stack or stack.pop() != pairs[ch]):
            return False
    return not stack

print(is_valid("{[()]}"))


True


### Checklist item: Monotonic stack

**Approach:**
- Why this matters: Finding the 'next greater element' for every position with nested loops is O(n^2); a monotonic stack exists specifically to answer that question for all positions in one linear pass by never re-examining a resolved element.
- Why this is the optimal approach: Each index is pushed onto the stack exactly once and popped at most once, so total stack operations across the whole array are bounded by 2n, giving O(n) amortized time — optimal because you must at least look at every element once, and the monotonic invariant guarantees no element is examined more than a constant number of times.
- Recognize the pattern: Maintain a stack of indices whose answer is still unknown, keeping the values monotonic (decreasing here); when a new value is greater than the value at the top index, that index's answer is resolved and it's popped.
- Code walkthrough:
- Maintains a stack of indices whose 'next greater' value is still unknown.
- For each new value, pops every index whose stored value is smaller, recording the current `value` as that index's next-greater answer.
- Indices left on the stack at the end have no greater element to their right and keep their initial `-1`.

**Learn more:**
- Website: [GeeksforGeeks: Stack Data Structure](https://www.geeksforgeeks.org/stack-data-structure/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Monotonic+stack+data+structures+algorithms)

**Trade-offs:**
- A monotonic stack solves 'next greater/smaller element' in O(n), versus O(n^2) for the brute-force nested loop — the trade is a modest increase in code complexity for an asymptotic win that matters once n is large.
- This implementation finds the *next* greater element (looking forward); flipping the iteration direction (right to left) instead finds the *previous* greater element — the same technique, different iteration order, so know which one the problem actually asks for.

**Practical software engineering use cases:**
- When to use it: Use a monotonic stack for next-greater/next-smaller, largest-rectangle-in-histogram, or stock-span style problems where each answer depends on the nearest qualifying neighbor.
- When not to use it: Do not reach for a monotonic stack when you just need the global max/min — that's a single O(n) scan with no stack required.


In [18]:
# Next greater element using a monotonic decreasing stack.
nums = [2, 1, 2, 4, 3]
answer = [-1] * len(nums)
stack = []
for i, value in enumerate(nums):
    while stack and nums[stack[-1]] < value:
        answer[stack.pop()] = value
    stack.append(i)
print(answer)


[4, 2, 4, -1, -1]


### Checklist item: BFS queue usage

**Approach:**
- Why this matters: Using a queue for BFS is not a stylistic choice — it's the mechanism that guarantees nodes are visited in order of distance from the source; swapping the queue for a stack turns the traversal into DFS and changes which node is discovered first.
- Why this is the optimal approach: Enqueuing all neighbors of the current frontier before dequeuing any of them processes the graph strictly layer by layer, so the first time a node is dequeued is guaranteed to be via a shortest path in an unweighted graph — O(V + E) time because each vertex and edge is processed once, which is optimal since you must at least look at every reachable vertex and edge once.
- Recognize the pattern: Put starting states into the queue, mark them seen immediately (at enqueue time, not dequeue time, to avoid enqueuing duplicates), then process one frontier layer at a time.
- Code walkthrough:
- Initialises the queue with `'A'` and a `seen` set to prevent revisiting nodes.
- Dequeues one node at a time, records it in `order`, then enqueues each unseen neighbour.
- Produces `['A', 'B', 'C', 'D']` — breadth-first order, visiting all nodes at distance 1 before distance 2.

**Learn more:**
- Website: [cp-algorithms: Breadth First Search](https://cp-algorithms.com/graph/breadth-first-search.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=BFS+queue+usage+data+structures+algorithms)

**Trade-offs:**
- Marking a node as seen at enqueue time (rather than dequeue time) prevents the same node from being added to the queue multiple times by different neighbors — skipping this is a common bug that causes redundant work or infinite loops on graphs with cycles.
- BFS guarantees shortest paths only when all edges have equal weight; for weighted graphs you need Dijkstra's algorithm (a priority queue instead of a plain queue) to get the same shortest-path guarantee.

**Practical software engineering use cases:**
- When to use it: Use BFS with a queue for shortest-path-in-hops problems, level-order traversal, or finding the minimum number of steps between states.
- When not to use it: Do not use plain BFS on weighted graphs where edge costs differ — use Dijkstra's algorithm instead so the priority queue accounts for cumulative path cost, not just hop count.


In [19]:
# BFS uses a queue to visit nodes by distance.
from collections import deque
graph = {"A": ["B", "C"], "B": ["D"], "C": [], "D": []}
queue = deque(["A"])
seen = {"A"}
order = []
while queue:
    node = queue.popleft()
    order.append(node)
    for neighbor in graph[node]:
        if neighbor not in seen:
            seen.add(neighbor)
            queue.append(neighbor)
print(order)


['A', 'B', 'C', 'D']


### Checklist item: Common problems: Valid Parentheses, Min Stack, Daily Temperatures, Evaluate Reverse Polish Notation

**Approach:**
- Why this matters: This item exists to turn stack pattern-recognition into a working solution on a genuinely different problem (expression evaluation) than the bracket-matching and monotonic-stack items already covered, so you practice recognizing when a stack — not a bracket check — is the right tool.
- Why this is the optimal approach: Evaluating Reverse Polish Notation with a single stack pass is O(n) time and O(n) worst-case space: each token is processed exactly once, operands are pushed, and each operator pops exactly two operands and pushes one result — optimal because you must read every token at least once, and the stack gives O(1) access to the two most recently computed values, which is exactly what postfix notation requires.
- Recognize the pattern: Use the list as a practice queue: pick one problem, write the brute-force version first (recursively parse infix-style expressions), identify why postfix removes the need for parentheses and precedence rules entirely, then implement the stack-based evaluator.
- Code walkthrough:
- Defines `eval_rpn(tokens)`: maintains one stack; for each token, if it's an operator, pops the two most recent operands (`b` then `a`, preserving order), applies the operator, and pushes the result back.
- Non-operator tokens are parsed as integers and pushed directly onto the stack.
- On `['2', '1', '+', '3', '*']`, computes `2+1=3` first, pushes `3`, then `3*3=9`, matching the postfix expression `(2+1)*3`.

**Learn more:**
- Website: [GeeksforGeeks: Stack Data Structure](https://www.geeksforgeeks.org/stack-data-structure/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Valid+Parentheses+data+structures+algorithms)

**Trade-offs:**
- This implementation uses truncating integer division (`int(a / b)`) to match typical RPN-calculator semantics for negative results; Python's `//` floor-divides instead, which gives a different (and wrong, for this problem) result on negative numbers.
- A single stack is O(n) space in the worst case (all operands pushed before any operator); if memory were a genuine constraint you could evaluate a fully-parenthesized infix expression directly without ever materializing a full token stack, at the cost of a more complex parser.

**Practical software engineering use cases:**
- When to use it: Use a stack for evaluating postfix/prefix expressions, implementing a calculator, or any grammar where operands must wait for their operator to arrive.
- When not to use it: Don't use this pattern for infix expressions with precedence and parentheses directly — either convert to postfix first (shunting-yard algorithm) or use a recursive-descent parser.


In [ ]:
# Stacks and Queues - Common problems: Valid Parentheses, Min Stack, Daily Temperatures, Evaluate Reverse Polish Notation
# Evaluate Reverse Polish Notation solved in full below; the remaining problems stay on the practice queue.
def eval_rpn(tokens):
    stack = []
    ops = {
        "+": lambda a, b: a + b,
        "-": lambda a, b: a - b,
        "*": lambda a, b: a * b,
        "/": lambda a, b: int(a / b),
    }
    for token in tokens:
        if token in ops:
            b = stack.pop()
            a = stack.pop()
            stack.append(ops[token](a, b))
        else:
            stack.append(int(token))
    return stack[0]

print(eval_rpn(["2", "1", "+", "3", "*"]))  # 9, from (2 + 1) * 3

practice_queue = [
    {'problem': 'Valid Parentheses', 'topic': 'Stacks and Queues', 'status': 'todo'},
    {'problem': 'Min Stack', 'topic': 'Stacks and Queues', 'status': 'todo'},
    {'problem': 'Daily Temperatures', 'topic': 'Stacks and Queues', 'status': 'todo'}
]

for entry in practice_queue:
    print(f"{entry['topic']}: {entry['problem']} -> {entry['status']}")


In [21]:
# Practice: Stacks and Queues

# Problem:
# Approach:
# Time Complexity:
# Space Complexity:
# Edge Cases:



## Linked Lists

### Study checklist
- [ ] Singly linked list traversal
- [ ] Fast and slow pointers
- [ ] Reversing a linked list
- [ ] Detecting cycles
- [ ] Merging linked lists
- [ ] Common problems: Reverse Linked List, Merge Two Sorted Lists, Linked List Cycle, Remove Nth Node From End

### Notes
Write your understanding, patterns, mistakes, and edge cases here.


### Beginner-friendly intro
A linked list is a chain of nodes where each node holds a value and a pointer to the next node.
You use them when you want cheap insert/delete in the middle of a sequence without shifting many elements.
Interview questions often focus on careful pointer movement (like fast/slow pointers) rather than index arithmetic.


### Checklist item: Singly linked list traversal

**Approach:**
- Why this matters: A linked list has no index — `head[3]` doesn't exist — so traversal by following `.next` references is the only way to reach any node, and getting the loop-termination condition wrong (using `current` instead of `current.next` somewhere) is the most common source of AttributeError-on-None bugs.
- Why this is the optimal approach: Traversal is O(n) time and O(1) extra space because each node is visited exactly once via its stored reference — this is the theoretical minimum, since you cannot know a node exists without following the chain to reach it (unlike an array, there's no way to skip ahead).
- Recognize the pattern: Track node references explicitly: hold `current`, check `current is not None` before dereferencing, and advance with `current = current.next` — never reuse `current` for anything other than 'the node I'm currently examining'.
- Code walkthrough:
- Defines a minimal `Node` class with `value` and `next`, then chains three nodes to form `1 → 2 → 3`.
- Walks the list with `current = current.next` until `current` is `None`, printing each value.

**Learn more:**
- Website: [GeeksforGeeks: Linked List Data Structure](https://www.geeksforgeeks.org/data-structures/linked-list/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Singly+linked+list+traversal+data+structures+algorithms)

**Trade-offs:**
- Compared to an array, a linked list gives up O(1) random access (`current.next` must be followed one node at a time, so reaching the k-th node is O(k)) in exchange for O(1) insertion/deletion once you already hold the relevant node.
- This traversal only reads values; if you need to also remove nodes while traversing, you must keep a `prev` reference so you can relink `prev.next` — traversal alone doesn't give you that.

**Practical software engineering use cases:**
- When to use it: Use explicit traversal whenever you need to read, search, or print every node's value, or as the basis for more advanced operations like reversal or cycle detection.
- When not to use it: Do not use a linked list (or its traversal) when you need frequent random access by index — Python lists give O(1) indexing that a linked list cannot match.


In [22]:
# Traverse a singly linked list.
class Node:
    def __init__(self, value, next=None):
        self.value = value
        self.next = next

head = Node(1, Node(2, Node(3)))
current = head
while current:
    print(current.value)
    current = current.next


1
2
3


### Checklist item: Fast and slow pointers

**Approach:**
- Why this matters: Finding the middle of a linked list by counting length first requires two passes (one to count, one to walk to n//2); the fast/slow pointer technique finds it in a single pass, which matters when the list is a stream or too large to traverse twice cheaply.
- Why this is the optimal approach: Advancing `slow` by 1 and `fast` by 2 per step means that when `fast` reaches the end (having traveled n steps), `slow` has traveled exactly n/2 steps — this is optimal because it achieves the two-pass result (find length, then find midpoint) in a single O(n) pass with O(1) extra space, using the ratio of speeds instead of an explicit counter.
- Recognize the pattern: Track node references explicitly with two independently-advancing pointers; the invariant 'fast moves twice as far as slow' is what guarantees slow lands on the midpoint exactly when fast runs out of list.
- Code walkthrough:
- Starts both `slow` and `fast` at the head, then advances `slow` one step and `fast` two steps each iteration.
- When `fast` or `fast.next` is `None`, `slow` sits at the middle node.
- Prints `slow.value = 3` — the middle of a 4-node list rounds up to the second middle.

**Learn more:**
- Website: [GeeksforGeeks: Linked List Data Structure](https://www.geeksforgeeks.org/data-structures/linked-list/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Fast+and+slow+pointers+data+structures+algorithms)

**Trade-offs:**
- This finds the *second* middle node for even-length lists (fast becomes None exactly when slow is at position n/2, using 0-indexing); if the problem wants the *first* middle for even lengths, the loop condition needs `fast.next and fast.next.next` instead.
- Fast/slow pointers find the middle in one O(n) pass with O(1) space; converting to a Python list first and indexing `lst[len(lst)//2]` is easier to read but costs O(n) extra space to materialize the list.

**Practical software engineering use cases:**
- When to use it: Use fast/slow pointers to find the middle of a list, detect palindromes (find middle, reverse second half, compare), or split a list into two halves without counting first.
- When not to use it: Do not use this technique if you already know the list's length (or it's cheap to get, e.g., a Python list) — just index directly at `n // 2`.


In [23]:
# Fast/slow pointers find the middle of a linked list.
class Node:
    def __init__(self, value, next=None):
        self.value = value
        self.next = next

head = Node(1, Node(2, Node(3, Node(4))))
slow = fast = head
while fast and fast.next:
    slow = slow.next
    fast = fast.next.next
print(slow.value)


3


### Checklist item: Reversing a linked list

**Approach:**
- Why this matters: Reversing a linked list by building a new list would cost O(n) extra memory; doing it in place by relinking `.next` pointers is the standard technique because it demonstrates the core skill this whole section is about — safely mutating pointers without losing the rest of the chain.
- Why this is the optimal approach: Relinking each node's `.next` to point backward while tracking `prev` and saving `next` before overwriting is O(n) time and O(1) extra space, which is optimal because you must visit and rewrite every node's pointer at least once, and this does exactly that with no revisits.
- Recognize the pattern: Track node references explicitly: save `current.next` in a temporary variable *before* reassigning `current.next = prev`, since overwriting first would lose the rest of the list forever — this save-before-overwrite order is the one invariant that must never be skipped.
- Code walkthrough:
- Saves `current.next` in `nxt` before relinking, then reverses the arrow: `current.next = prev`.
- Advances both `prev` and `current` forward after each reversal; after the loop `prev` is the new head.
- Prints `prev.value = 3` — the former tail is now the first node.

**Learn more:**
- Website: [GeeksforGeeks: Linked List Data Structure](https://www.geeksforgeeks.org/data-structures/linked-list/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Reversing+a+linked+list+data+structures+algorithms)

**Trade-offs:**
- In-place reversal is O(1) extra space but destroys the original forward-order list; if another part of the program still holds a reference to the old head expecting forward order, it will now see a single-node fragment, since that node's `.next` now points backward.
- The iterative version here uses O(1) space; a recursive reversal is arguably more elegant but uses O(n) call-stack space and risks a stack overflow on very long lists, so prefer the iterative form for production code.

**Practical software engineering use cases:**
- When to use it: Use in-place reversal when memory matters and you don't need the original order preserved — implementing undo stacks, reversing traversal order, or as a subroutine in palindrome checks.
- When not to use it: Do not reverse in place if other references to the original list structure must remain valid in forward order; build a new reversed copy instead.


In [24]:
# Reverse a linked list iteratively.
class Node:
    def __init__(self, value, next=None):
        self.value = value
        self.next = next

head = Node(1, Node(2, Node(3)))
prev, current = None, head
while current:
    nxt = current.next
    current.next = prev
    prev = current
    current = nxt
print(prev.value)


3


### Checklist item: Detecting cycles

**Approach:**
- Why this matters: A cycle in a linked list breaks the assumption every other item in this section relies on — that traversal eventually reaches `None`; without cycle detection, a naive traversal loops forever and either hangs or crashes with a memory/stack issue.
- Why this is the optimal approach: Floyd's algorithm (fast/slow pointers) detects a cycle in O(n) time and O(1) space by the pigeonhole principle: once both pointers are inside the cycle, the fast pointer gains one step on the slow pointer every iteration, so it must catch up (meet it) within at most the cycle's length — this beats the alternative of storing every visited node in a set, which also detects cycles correctly but costs O(n) extra space.
- Recognize the pattern: Track node references explicitly with two pointers moving at different speeds; use `slow is fast` (identity comparison, not `==`) to detect when they land on the same node object, since two different nodes could coincidentally have equal `.value`.
- Code walkthrough:
- Constructs a cyclic list: `a → b → c → b` (c points back to b, creating a loop).
- Moves `slow` one step and `fast` two steps per iteration; `slow is fast` (identity, not equality) fires when they meet inside the cycle.
- Sets `has_cycle = True` and breaks as soon as the pointers meet.

**Learn more:**
- Website: [GeeksforGeeks: Linked List Data Structure](https://www.geeksforgeeks.org/data-structures/linked-list/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Detecting+cycles+data+structures+algorithms)

**Trade-offs:**
- Floyd's algorithm uses O(1) space versus O(n) for a visited-set approach, but it's harder to reason about correctness at a glance; the set-based approach is more intuitive to write correctly under interview pressure at the cost of memory.
- This detects *whether* a cycle exists but not *where* it begins; finding the cycle's start node requires an extra phase (reset one pointer to head, advance both one step at a time until they meet again) which is a well-known but separate extension.

**Practical software engineering use cases:**
- When to use it: Use Floyd's cycle detection whenever a linked structure might loop back on itself and you need to detect that fact in O(1) space — safe traversal, corruption detection, or the classic interview follow-up of finding the cycle's entry point.
- When not to use it: Do not use this technique on structures already known to be acyclic (like a validated tree) — the extra pointer bookkeeping is unnecessary overhead there.


In [25]:
# Floyd cycle detection.
class Node:
    def __init__(self, value):
        self.value = value
        self.next = None

a, b, c = Node("a"), Node("b"), Node("c")
a.next, b.next, c.next = b, c, b
slow = fast = a
has_cycle = False
while fast and fast.next:
    slow = slow.next
    fast = fast.next.next
    if slow is fast:
        has_cycle = True
        break
print(has_cycle)


True


### Checklist item: Merging linked lists

**Approach:**
- Why this matters: Merging two sorted sequences into one sorted sequence without re-sorting is the building block behind merge sort and behind combining sorted data streams (e.g., two sorted result sets) — re-sorting the concatenation would throw away the fact that both inputs were already sorted.
- Why this is the optimal approach: Advancing whichever pointer points to the smaller current element does O(1) work per element and never revisits an element, giving O(n + m) time for lists of length n and m — optimal because you must place every element from both lists into the output at least once, and this algorithm does exactly one comparison per placement.
- Recognize the pattern: Track pointer references explicitly on both sequences; compare the two current elements, append the smaller, advance only that one pointer, and when one sequence is exhausted, append the remainder of the other directly since it's already sorted.
- Code walkthrough:
- Uses index pointers `i` and `j` on two sorted Python lists to simulate the same merge logic used for linked list nodes.
- Compares `a[i]` and `b[j]`, appends the smaller, and advances only that pointer; after one list is exhausted, extends with the remaining tail.
- Produces the fully merged sorted list `[1, 2, 3, 4, 5, 6]`.

**Learn more:**
- Website: [GeeksforGeeks: Linked List Data Structure](https://www.geeksforgeeks.org/data-structures/linked-list/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Merging+linked+lists+data+structures+algorithms)

**Trade-offs:**
- Merging is O(n + m) versus concatenating then sorting, which is O((n+m) log(n+m)) — merging is strictly better whenever both inputs are already sorted, which is the whole point of using it instead of a generic sort.
- This builds a brand-new merged list; for real linked lists (not Python list simulation), you can merge by relinking existing nodes' `.next` pointers instead of allocating new nodes, which saves memory at the cost of mutating the original lists.

**Practical software engineering use cases:**
- When to use it: Use this to merge sorted linked lists, merge sorted external files/streams that don't fit in memory, or as the merge step of merge sort.
- When not to use it: Do not use a merge when the inputs aren't already sorted — sort them first (or use a different combining strategy), since merging unsorted inputs produces an unsorted result.


In [26]:
# Merge two sorted Python lists with the same pointer idea used for linked lists.
a = [1, 3, 5]
b = [2, 4, 6]
i = j = 0
merged = []
while i < len(a) and j < len(b):
    if a[i] <= b[j]:
        merged.append(a[i]); i += 1
    else:
        merged.append(b[j]); j += 1
merged.extend(a[i:])
merged.extend(b[j:])
print(merged)


[1, 2, 3, 4, 5, 6]


### Checklist item: Common problems: Reverse Linked List, Merge Two Sorted Lists, Linked List Cycle, Remove Nth Node From End

**Approach:**
- Why this matters: This item exists to turn linked-list pointer-manipulation skill into a working solution on a genuinely different problem (merging real Node objects) than the earlier items, which either used a single list or a Python-list simulation, so relinking actual node references gets real practice.
- Why this is the optimal approach: Merging with a dummy head node and a `tail` pointer that always points to the last placed node does O(1) work per node and O(n + m) total time — optimal because, as with the earlier merge item, every node from both lists must be placed at least once, and using existing nodes (rather than copying values into new ones) also keeps space at O(1) extra beyond the input lists.
- Recognize the pattern: Use the list as a practice queue: pick one problem, write the brute-force version first (collect all values into a Python list, sort, rebuild a linked list), identify the wasted O((n+m) log(n+m)) sort step, then replace it with a direct O(n+m) merge using two real Node chains.
- Code walkthrough:
- Defines `merge_two_lists(a, b)`: uses a `dummy` sentinel node so the first real node doesn't need special-casing, and a `tail` pointer that always points to the last node placed in the result.
- At each step, relinks `tail.next` to whichever of `a` or `b` currently holds the smaller value, then advances that source pointer — this reuses existing nodes instead of allocating new ones.
- Once one list is exhausted, `tail.next = a or b` attaches the remaining (already sorted) tail of the other list in O(1), since it needs no further comparison.

**Learn more:**
- Website: [GeeksforGeeks: Linked List Data Structure](https://www.geeksforgeeks.org/data-structures/linked-list/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Reverse+Linked+List+data+structures+algorithms)

**Trade-offs:**
- Using a dummy head node avoids special-casing 'is this the first node in the result', at the cost of one throwaway node and the need to return `dummy.next` instead of `dummy` itself — a small, standard idiom worth memorizing.
- This mutates and reuses the input nodes (relinking their `.next` pointers) rather than copying values into new nodes; that's more memory-efficient but means the original lists `a` and `b` are no longer valid as separate lists afterward.

**Practical software engineering use cases:**
- When to use it: Use node-relinking merges when combining sorted linked lists in place is required — merge sort's merge step, combining sorted result pages, or any scenario where allocating new nodes would be wasteful.
- When not to use it: Don't use this in-place relinking approach if the original lists must remain intact and usable afterward — copy values into new nodes instead.


In [ ]:
# Linked Lists - Common problems: Reverse Linked List, Merge Two Sorted Lists, Linked List Cycle, Remove Nth Node From End
# Merge Two Sorted Lists solved in full below; the remaining problems stay on the practice queue.
class ListNode:
    def __init__(self, val, next=None):
        self.val = val
        self.next = next

def merge_two_lists(a, b):
    dummy = ListNode(0)
    tail = dummy
    while a and b:
        if a.val <= b.val:
            tail.next, a = a, a.next
        else:
            tail.next, b = b, b.next
        tail = tail.next
    tail.next = a or b
    return dummy.next

def list_to_python(head):
    out = []
    while head:
        out.append(head.val)
        head = head.next
    return out

l1 = ListNode(1, ListNode(3, ListNode(5)))
l2 = ListNode(2, ListNode(4, ListNode(6)))
print(list_to_python(merge_two_lists(l1, l2)))  # [1, 2, 3, 4, 5, 6]

practice_queue = [
    {'problem': 'Reverse Linked List', 'topic': 'Linked Lists', 'status': 'todo'},
    {'problem': 'Linked List Cycle', 'topic': 'Linked Lists', 'status': 'todo'},
    {'problem': 'Remove Nth Node From End', 'topic': 'Linked Lists', 'status': 'todo'}
]

for entry in practice_queue:
    print(f"{entry['topic']}: {entry['problem']} -> {entry['status']}")


In [28]:
# Practice: Linked Lists

# Problem:
# Approach:
# Time Complexity:
# Space Complexity:
# Edge Cases:



## Trees

### Study checklist
- [ ] Tree traversal: preorder, inorder, postorder
- [ ] Breadth-first traversal / level order
- [ ] Recursive tree reasoning
- [ ] Tree height and depth
- [ ] Common problems: Maximum Depth of Binary Tree, Same Tree, Invert Binary Tree, Binary Tree Level Order Traversal

### Notes
Write your understanding, patterns, mistakes, and edge cases here.


### Beginner-friendly intro
A tree is a hierarchical structure with a root at the top and children below, with no cycles.
You use trees to model parent-child relationships, file systems, or recursive decompositions of data.
Most problems here ask you to traverse the tree and compute something about nodes or paths.


### Checklist item: Tree traversal: preorder, inorder, postorder

**Approach:**
- Why this matters: Preorder, inorder, and postorder aren't three arbitrary variations — each ordering exposes a different fact about the tree (preorder recreates construction order, inorder gives sorted order for a BST, postorder guarantees children are processed before parents), so picking the wrong one for a task means redoing the work.
- Why this is the optimal approach: Recursing into every child once and combining results is O(n) time (each of n nodes is visited exactly once) and O(h) additional call-stack space for a tree of height h — optimal because any traversal that must report every node's value cannot do better than visiting each node once.
- Recognize the pattern: Start from the root, recurse into left and right children (or all children, for a general tree), then combine their results at the current node — the *position* of that combine step relative to the recursive calls is what defines pre/in/post-order.
- Code walkthrough:
- Represents the tree as an adjacency dict where each key maps to a list of child nodes.
- `preorder` places the current node first, then `extend`s with each child's preorder result recursively.
- Returns `['A', 'B', 'D', 'E', 'C']` — root appears before its subtrees (pre-order: node, left, right).

**Learn more:**
- Website: [GeeksforGeeks: Binary Tree Data Structure](https://www.geeksforgeeks.org/binary-tree-data-structure/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Tree+traversal+data+structures+algorithms)

**Trade-offs:**
- Preorder here visits the current node before its children (`result = [node]` then extend), which reconstructs a valid build order; if you need sorted output from a BST instead, inorder (left, node, right) is the correct choice, not preorder.
- This recursive version is easy to read but uses O(h) call-stack space and risks a stack overflow on very deep, unbalanced trees; an iterative version using an explicit stack avoids recursion depth limits at the cost of more verbose code.

**Practical software engineering use cases:**
- When to use it: Use preorder to serialize/clone a tree (parent before children), inorder to read a BST in sorted order, and postorder to safely delete or evaluate a tree bottom-up (children before parent).
- When not to use it: Do not use recursive traversal on trees that might be extremely deep (e.g., untrusted or adversarial input) without converting to an iterative version — deep recursion can exceed Python's stack limit.


In [29]:
# Tree traversal.
tree = {"A": ["B", "C"], "B": ["D", "E"], "C": [], "D": [], "E": []}

def preorder(node):
    result = [node]
    for child in tree[node]:
        result.extend(preorder(child))
    return result

print(preorder("A"))


['A', 'B', 'D', 'E', 'C']


### Checklist item: Breadth-first traversal / level order

**Approach:**
- Why this matters: Level-order traversal answers a fundamentally different question than depth-first orders (preorder/inorder/postorder): 'what does the tree look like level by level', which matters for anything based on distance-from-root, like finding the minimum depth or printing a tree visually.
- Why this is the optimal approach: Using a queue and extending it with each node's children processes the tree in strict breadth-first order, visiting each node exactly once for O(n) time and O(w) space where w is the widest level — optimal because reporting nodes grouped by level requires knowing when one level ends and the next begins, which only a FIFO queue naturally provides (a stack would give depth-first order instead).
- Recognize the pattern: Put the root into a queue, then repeatedly dequeue a node and enqueue its children — dequeuing before enqueuing children preserves the guarantee that a queue naturally batches nodes by depth, since all of depth d are enqueued (by nodes at depth d-1) before any of depth d+1.
- Code walkthrough:
- Initialises a queue with root `'A'` and dequeues one node at a time, printing it immediately.
- Extends the queue with the node's children list directly, so all children of the current level are enqueued before any grandchildren.
- No `seen` set is needed here because a tree has no back edges — each node has exactly one parent.

**Learn more:**
- Website: [GeeksforGeeks: Binary Tree Data Structure](https://www.geeksforgeeks.org/binary-tree-data-structure/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Breadth-first+traversal+%2F+level+order+data+structures+algorithms)

**Trade-offs:**
- Unlike level order on a general graph, a tree traversal never needs a `seen` set — because a tree has no back edges or shared ancestors, a node cannot be reached twice, so the extra bookkeeping a graph BFS requires is unnecessary here.
- This implementation prints nodes as they're dequeued without separating levels; if you need each level as its own list (e.g., for level-order output as a list of lists), you must snapshot `len(queue)` at the start of each level's processing before mutating the queue further.

**Practical software engineering use cases:**
- When to use it: Use level-order traversal for printing a tree row by row, finding the minimum depth (BFS reaches the nearest leaf first), or serializing a tree in a way that preserves level structure.
- When not to use it: Do not use level order when you need ancestor/descendant relationships or a full root-to-leaf path — depth-first traversals naturally carry that path information as the call stack unwinds; level order does not.


In [30]:
# Level-order traversal.
from collections import deque
tree = {"A": ["B", "C"], "B": ["D"], "C": [], "D": []}
queue = deque(["A"])
while queue:
    node = queue.popleft()
    print(node)
    queue.extend(tree[node])


A
B
C
D


### Checklist item: Recursive tree reasoning

**Approach:**
- Why this matters: Recursive tree reasoning is the mental model behind nearly every tree algorithm: trusting that a recursive call correctly answers the question for a subtree, then combining those answers, is what lets you solve problems on trees of unknown, unbounded shape without hardcoding depth.
- Why this is the optimal approach: Each node makes exactly one recursive call per child and does O(1) work combining the results (here, summing sizes), giving O(n) total time across all n nodes — optimal because computing any whole-tree aggregate requires visiting every node at least once, and this recursion visits each node exactly once.
- Recognize the pattern: Trust the recursive call to correctly answer the question for each child subtree (the 'leap of faith'), then combine those answers at the current node with O(1) extra work — this is the general recursive-tree-reasoning pattern, here applied to counting nodes.
- Code walkthrough:
- `subtree_size` returns `1` (itself) plus the sum of `subtree_size` called on every child recursively.
- Bottoms out at leaf nodes whose children list is empty, each contributing exactly `1`.
- Prints `4` — the total node count of the tree rooted at `'A'`.

**Learn more:**
- Website: [GeeksforGeeks: Binary Tree Data Structure](https://www.geeksforgeeks.org/binary-tree-data-structure/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Recursive+tree+reasoning+data+structures+algorithms)

**Trade-offs:**
- This recursion is simple because subtree results don't depend on anything outside the subtree (no parent context needed); a problem like 'diameter of a tree' additionally needs to track a value across the whole tree, which requires either a mutable accumulator or returning a tuple of results from each call.
- `sum(subtree_size(child) for child in tree[node])` recomputes nothing — each subtree's size is computed exactly once and reused in its parent's sum, so there's no memoization needed here (unlike DP-on-trees problems where the same subtree answer might be needed by multiple callers).

**Practical software engineering use cases:**
- When to use it: Use bottom-up recursive aggregation for counting nodes, summing values, computing subtree properties, or any 'answer for this node depends only on answers for its children' problem.
- When not to use it: Do not use plain bottom-up recursion when a node's answer also depends on information from outside its subtree (its ancestors or siblings) — that requires passing context downward or a two-pass rerooting technique instead.


In [31]:
# Each call returns the size of its subtree.
tree = {"A": ["B", "C"], "B": ["D"], "C": [], "D": []}
def subtree_size(node):
    return 1 + sum(subtree_size(child) for child in tree[node])
print(subtree_size("A"))


4


### Checklist item: Tree height and depth

**Approach:**
- Why this matters: Height and depth are the two measurements every balance-related tree algorithm depends on (AVL rotations, balanced-tree validation, recursion-depth estimation); confusing 'height from this node down' with 'depth from the root down to this node' is a common source of off-by-one bugs.
- Why this is the optimal approach: Computing `1 + max(height(child) for child in children)` bottom-up visits each node exactly once and does O(1) work per node beyond the recursive calls, giving O(n) total time — optimal because determining the height of the whole tree requires knowing the height of every subtree, so no algorithm can avoid visiting all n nodes.
- Recognize the pattern: Recurse to the leaves first (base case: a leaf has height 1), then combine each node's children's heights by taking the max and adding 1 for the current node's own level — this bottom-up combination is what distinguishes height computation from a simple traversal.
- Code walkthrough:
- Returns `1` for any leaf (empty children list), then recursively computes `1 + max(height(child) for child in children)`.
- The deepest path `A → B → D` has three nodes, so `height('A')` returns `3`.

**Learn more:**
- Website: [GeeksforGeeks: Binary Tree Data Structure](https://www.geeksforgeeks.org/binary-tree-data-structure/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Tree+height+and+depth+data+structures+algorithms)

**Trade-offs:**
- This computes *height* (longest path down to a leaf, counted in nodes); *depth* is the distance from the root down to this specific node, computed top-down instead — the two are easy to conflate but require opposite recursion directions.
- This treats a leaf (empty children list) as height 1; some definitions instead treat height as edge-count (a single node has height 0) — the constant offset doesn't change the algorithm's O(n) complexity, but it does change every comparison against a specific height value, so pick one convention and stay consistent.

**Practical software engineering use cases:**
- When to use it: Use height computation to validate whether a tree is balanced (compare left/right subtree heights at every node), to estimate recursion-depth risk, or to compute the diameter of a tree.
- When not to use it: Do not recompute height repeatedly from scratch for every node if you need it for many nodes — compute it bottom-up once, in a single O(n) pass, and cache each subtree's height as you go.


In [32]:
# Height/depth by recursive child aggregation.
tree = {"A": ["B", "C"], "B": ["D"], "C": [], "D": []}
def height(node):
    if not tree[node]:
        return 1
    return 1 + max(height(child) for child in tree[node])
print(height("A"))


3


### Checklist item: Common problems: Maximum Depth of Binary Tree, Same Tree, Invert Binary Tree, Binary Tree Level Order Traversal

**Approach:**
- Why this matters: This item exists to turn tree-recursion pattern recognition into a working solution on a genuinely different problem (structural mutation) than the traversal, sizing, and height items already covered, so 'return a modified tree, not just a value' gets real practice.
- Why this is the optimal approach: Inverting a binary tree by swapping `node.left` and `node.right` at every node, recursively, does O(1) work per node and visits each of the n nodes exactly once, giving O(n) time — optimal because producing a mirrored tree requires touching every node's child pointers at least once, and this algorithm does exactly that with no revisits.
- Recognize the pattern: Use the list as a practice queue: pick one problem, write the brute-force version first (build a brand-new mirrored tree by copying nodes), identify that swapping existing child pointers in place avoids that copy entirely, then implement the in-place swap recursively.
- Code walkthrough:
- Defines `invert_tree(node)`: at each node, recursively inverts the right and left subtrees first, then swaps which inverted subtree becomes `node.left` vs `node.right`.
- The base case `if node is None: return None` stops recursion at leaves' empty children without special-casing them in the caller.
- `preorder_values` confirms the mutation: before inversion the tree reads `[4, 2, 1, 3, 7, 6, 9]`; after, the left/right children are swapped at every level, producing `[4, 7, 9, 6, 2, 3, 1]`.

**Learn more:**
- Website: [GeeksforGeeks: Binary Tree Data Structure](https://www.geeksforgeeks.org/binary-tree-data-structure/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Maximum+Depth+of+Binary+Tree+data+structures+algorithms)

**Trade-offs:**
- This mutates the tree in place (O(1) extra space beyond the recursion stack); building a brand-new mirrored tree instead would use O(n) extra memory but leave the original tree untouched, which matters if other code still holds references expecting the original structure.
- The recursive swap here reads as `node.left, node.right = invert_tree(node.right), invert_tree(node.left)` — the order of evaluation matters: Python evaluates the right-hand side fully before assigning, so this correctly captures both recursive results before either pointer is overwritten.

**Practical software engineering use cases:**
- When to use it: Use in-place recursive mutation for structural transformations of a tree: inverting, pruning subtrees, or relabeling nodes based on their position.
- When not to use it: Don't mutate in place if the original tree must remain valid for other consumers — return a new tree instead, accepting the O(n) extra memory cost.


In [ ]:
# Trees - Common problems: Maximum Depth of Binary Tree, Same Tree, Invert Binary Tree, Binary Tree Level Order Traversal
# Invert Binary Tree solved in full below; the remaining problems stay on the practice queue.
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def invert_tree(node):
    if node is None:
        return None
    node.left, node.right = invert_tree(node.right), invert_tree(node.left)
    return node

def preorder_values(node):
    if node is None:
        return []
    return [node.val] + preorder_values(node.left) + preorder_values(node.right)

tree = TreeNode(4, TreeNode(2, TreeNode(1), TreeNode(3)), TreeNode(7, TreeNode(6), TreeNode(9)))
print(preorder_values(tree))   # [4, 2, 1, 3, 7, 6, 9] before inversion
invert_tree(tree)
print(preorder_values(tree))   # [4, 7, 9, 6, 2, 3, 1] after inversion

practice_queue = [
    {'problem': 'Maximum Depth of Binary Tree', 'topic': 'Trees', 'status': 'todo'},
    {'problem': 'Same Tree', 'topic': 'Trees', 'status': 'todo'},
    {'problem': 'Binary Tree Level Order Traversal', 'topic': 'Trees', 'status': 'todo'}
]

for entry in practice_queue:
    print(f"{entry['topic']}: {entry['problem']} -> {entry['status']}")


In [34]:
# Practice: Trees

# Problem:
# Approach:
# Time Complexity:
# Space Complexity:
# Edge Cases:



## Binary Search Trees

### Study checklist
- [ ] BST property
- [ ] Search, insert, and delete
- [ ] Lowest common ancestor
- [ ] Validate BST
- [ ] Common problems: Validate BST, Kth Smallest Element in BST, Lowest Common Ancestor of BST

### Notes
Write your understanding, patterns, mistakes, and edge cases here.


### Beginner-friendly intro
A binary search tree is a tree where every value in the left subtree is smaller and every value in the right subtree is larger, based on the duplicate-value rule the implementation chooses.
Because of this ordering, you can search, insert, and delete while always moving left or right based on comparisons.
BST questions focus on using this ordering property to make operations faster than scanning the whole tree.


### Checklist item: BST property

**Approach:**
- Why this matters: The BST ordering property (everything left is smaller, everything right is larger) is the single invariant that makes search, insert, and delete all O(log n) on a balanced tree instead of O(n); every other BST item in this section depends on this property actually holding.
- Why this is the optimal approach: Checking `low < node.value < high` while narrowing the bounds on each recursive call does O(1) work per node and visits each node at most once, giving O(n) time to validate an entire tree — optimal because confirming the property holds everywhere requires examining every node at least once; there's no way to validate a global property without touching all n nodes.
- Recognize the pattern: Carry the BST ordering bounds through the recursion: each node must fall strictly between the bounds inherited from its ancestors, and recursing left tightens the upper bound while recursing right tightens the lower bound.
- Code walkthrough:
- Passes a narrowing `(low, high)` interval down the tree: every node must satisfy `low < node.value < high`.
- Recursing left tightens the upper bound to the parent's value; recursing right tightens the lower bound.
- Returns `True` for the valid BST `Node(5, Node(3), Node(8))`.

**Learn more:**
- Website: [GeeksforGeeks: Binary Search Tree](https://www.geeksforgeeks.org/binary-search-tree-data-structure/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=BST+property+data+structures+algorithms)

**Trade-offs:**
- A common but wrong shortcut is to only check `node.left.value < node.value < node.right.value` locally at each node; that misses violations from a grandchild that's in range for its immediate parent but out of range for a higher ancestor, which is exactly why the bounds must be threaded through the whole recursion.
- This validation is O(n) time and O(h) space (recursion stack, h = height); an alternative using inorder traversal and checking strictly-increasing order (the technique used later in this section) is equally O(n) but trades the bounds-threading logic for a single running 'previous value' check.

**Practical software engineering use cases:**
- When to use it: Validate the BST property whenever you receive a tree from an untrusted source, after a sequence of manual mutations, or before running an algorithm (like BST search) that silently gives wrong answers on a broken tree.
- When not to use it: Don't validate on every operation in a hot path if inserts/deletes are always performed through a trusted, tested API that maintains the invariant — validate at trust boundaries, not everywhere.


In [35]:
# Validate BST with lower/upper bounds.
class Node:
    def __init__(self, value, left=None, right=None):
        self.value = value
        self.left = left
        self.right = right

root = Node(5, Node(3), Node(8))
def is_bst(node, low=float("-inf"), high=float("inf")):
    if node is None:
        return True
    return low < node.value < high and is_bst(node.left, low, node.value) and is_bst(node.right, node.value, high)
print(is_bst(root))


True


### Checklist item: Search, insert, and delete

**Approach:**
- Why this matters: Search, insert, and delete are the three operations a BST exists to make fast; getting insert wrong (e.g., not returning the new subtree root back up the call chain) silently produces a tree where some inserted values are unreachable by search.
- Why this is the optimal approach: Because the BST property lets you discard an entire subtree at each comparison (go left or right, never both), each operation does O(h) work where h is the tree's height — O(log n) on a balanced tree, which is optimal because a comparison-based search over n ordered items cannot do better than O(log n) (matching binary search on a sorted array), while an unsorted structure would need O(n).
- Recognize the pattern: Compare the target value against the current node to decide whether to recurse left or right, and when you reach a `None` slot, that's exactly where a new node belongs — returning the (possibly newly created) node back up the call chain rewires the parent's pointer correctly.
- Code walkthrough:
- `insert` navigates left or right based on value comparison and creates a new `Node` when it reaches `None`, returning it up the call stack.
- Inserts `5, 3, 8, 4` in sequence; `4` lands as the right child of `3` (left subtree of root `5`).
- Prints `root.left.right.value = 4`, confirming the node was placed correctly.

**Learn more:**
- Website: [GeeksforGeeks: Binary Search Tree](https://www.geeksforgeeks.org/binary-search-tree-data-structure/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Search+data+structures+algorithms)

**Trade-offs:**
- The O(log n) bound depends entirely on the tree staying roughly balanced; a BST built by inserting already-sorted data degrades into a linked list with O(n) operations, which is why production systems use self-balancing variants (AVL, red-black trees) instead of a plain BST.
- Deletion is intentionally left out here because it has three cases (no children, one child, two children) — the two-children case in particular requires finding an in-order successor/predecessor to replace the removed node, which is meaningfully more complex than search or insert.

**Practical software engineering use cases:**
- When to use it: Use BST search/insert for maintaining a dynamically-changing sorted collection with fast lookups — leaderboards, indexed records, or interval scheduling.
- When not to use it: Don't use a plain (potentially unbalanced) BST when insert order is adversarial or already sorted — use a self-balancing tree or a library-backed structure to guarantee O(log n) regardless of insertion order.


In [36]:
# BST search and insert. Deletion has more cases, so practice it separately.
class Node:
    def __init__(self, value):
        self.value = value
        self.left = None
        self.right = None

def insert(node, value):
    if node is None:
        return Node(value)
    if value < node.value:
        node.left = insert(node.left, value)
    elif value > node.value:
        node.right = insert(node.right, value)
    return node

root = None
for value in [5, 3, 8, 4]:
    root = insert(root, value)
print(root.left.right.value)


4


### Checklist item: Lowest common ancestor

**Approach:**
- Why this matters: Finding the lowest common ancestor naively (find both root-to-node paths, then compare them) takes O(n) extra space for the paths; the BST-specific approach uses the ordering property to navigate directly toward the split point without storing any path.
- Why this is the optimal approach: Moving left when both targets are smaller than the current node and right when both are larger does O(1) work per step and stops the moment the targets diverge, giving O(h) time and O(1) extra space — optimal because the LCA is, by definition, the first node where the two search paths diverge, and following the BST ordering is guaranteed to trace exactly that path with no wasted steps.
- Recognize the pattern: Carry the BST ordering bounds implicitly by comparing both targets against the current node: if both are on the same side, the LCA must be deeper on that side; the instant they're on opposite sides (or one equals the current node), you've found the LCA.
- Code walkthrough:
- Walks the tree iteratively: moves left if both targets are smaller than the current node, right if both are larger.
- Returns the current node's value the moment the two targets diverge to opposite subtrees — that is the LCA.
- Prints `5` — the lowest common ancestor of `3` and `8` in this tree.

**Learn more:**
- Website: [GeeksforGeeks: Binary Search Tree](https://www.geeksforgeeks.org/binary-search-tree-data-structure/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Lowest+common+ancestor+data+structures+algorithms)

**Trade-offs:**
- This BST-specific LCA is O(h) time and O(1) space; the general-tree LCA algorithm (which doesn't assume ordering) needs O(n) time and typically O(h) extra space to search both subtrees explicitly, since it can't use value comparisons to prune the search.
- This iterative version needs O(1) extra space; a recursive version expresses the same logic more concisely but costs O(h) call-stack space, which only matters for extremely unbalanced trees.

**Practical software engineering use cases:**
- When to use it: Use the BST-ordering LCA whenever the tree is a genuine BST and you need the common ancestor of two known values, e.g., finding the shared category in a hierarchical taxonomy.
- When not to use it: Don't use this ordering-based shortcut on a general (non-BST) binary tree — without the ordering property, you must fall back to the general two-subtree-search LCA algorithm.


In [37]:
# LCA in a BST uses ordering to move toward the split point.
class Node:
    def __init__(self, value, left=None, right=None):
        self.value = value
        self.left = left
        self.right = right

root = Node(5, Node(3), Node(8))
def lca_value(root, a, b):
    node = root
    while node:
        if a < node.value and b < node.value:
            node = node.left
        elif a > node.value and b > node.value:
            node = node.right
        else:
            return node.value

print(lca_value(root, 3, 8))


5


### Checklist item: Validate BST

**Approach:**
- Why this matters: Validate BST reappears here using a different technique (inorder traversal) than the bounds-passing approach earlier in this section, because recognizing that a valid BST's inorder traversal is strictly increasing is a distinct, equally important insight — many BST problems reduce to 'run an inorder traversal and reason about the resulting sequence'.
- Why this is the optimal approach: Tracking only the previously-visited value (`prev`) while performing a standard inorder traversal does O(1) extra comparison work per node, giving O(n) time and O(h) space overall — this is optimal for the same reason as the bounds-passing approach (validating a global property requires visiting every node), but it trades explicit bound-tracking for the simpler invariant that inorder traversal of a valid BST visits values in strictly increasing order.
- Recognize the pattern: Use inorder traversal (left, node, right) to expose the BST's sorted order: if any visited value is not strictly greater than the previous one, the ordering property has been violated somewhere in the tree.
- Code walkthrough:
- Defines `is_bst_inorder(root)`: performs a standard inorder traversal, comparing each visited node's value against `prev[0]` (the last value seen) using a one-element list as a mutable closure variable.
- If `node.val <= prev[0]`, the sequence isn't strictly increasing, so the function short-circuits and returns `False` immediately without visiting the rest of the tree.
- On `TreeNode(5, TreeNode(3), TreeNode(7))` the inorder sequence is `3, 5, 7` (strictly increasing) so it prints `True`; on `TreeNode(5, TreeNode(3), TreeNode(4))` the sequence is `3, 5, 4` — `4` after `5` breaks the increasing order, so it prints `False`.

**Learn more:**
- Website: [GeeksforGeeks: Binary Search Tree](https://www.geeksforgeeks.org/binary-search-tree-data-structure/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Validate+BST+data+structures+algorithms)

**Trade-offs:**
- Using a one-element list (`prev = [float('-inf')]`) as a mutable cell is a common Python idiom to work around the lack of writable closures in nested functions; a cleaner alternative in modern Python is the `nonlocal` keyword, which achieves the same mutable-outer-variable effect with clearer intent.
- This inorder-based check is elegant but must complete the traversal (or short-circuit early) to know the answer; the bounds-passing approach from earlier in this section can, in principle, be adapted to prune more aggressively in some implementations, though both are O(n) in the worst case.

**Practical software engineering use cases:**
- When to use it: Use the inorder-strictly-increasing check whenever you're already performing an inorder traversal for another reason (like extracting sorted values) and want the validation essentially for free.
- When not to use it: Don't use this check if duplicate values are allowed in the tree by design — strict inequality (`<=`) will reject a valid tree that permits duplicates; use `<` for a duplicates-allowed variant if that's the intended semantics.


In [38]:
# Validate BST: inorder traversal must be strictly increasing.
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val = val; self.left = left; self.right = right

def is_bst_inorder(root):
    prev = [float('-inf')]          # last visited value
    def inorder(node):
        if not node:
            return True
        if not inorder(node.left):
            return False
        if node.val <= prev[0]:     # not strictly increasing -> invalid
            return False
        prev[0] = node.val
        return inorder(node.right)
    return inorder(root)

#     5          inorder: 3 5 7  (strictly increasing)  -> True
#    / \
#   3   7
valid = TreeNode(5, TreeNode(3), TreeNode(7))
print(is_bst_inorder(valid))    # True

#     5          inorder: 3 5 4  (5 > 4, not increasing) -> False
#    / \
#   3   4
invalid = TreeNode(5, TreeNode(3), TreeNode(4))
print(is_bst_inorder(invalid))  # False


{'topic': 'Binary Search Trees', 'checklist_item': 'Validate BST', 'steps': ['write a brute-force solution', 'name the key invariant or data structure', 'test empty, single-item, duplicate, and large inputs']}


### Checklist item: Common problems: Validate BST, Kth Smallest Element in BST, Lowest Common Ancestor of BST

**Approach:**
- Why this matters: This item exists to turn BST pattern-recognition into a working solution on a genuinely different problem (order statistics) than the search/insert, LCA, and validation items already covered, so 'inorder traversal gives sorted order, and the k-th visited node is the k-th smallest' gets real practice.
- Why this is the optimal approach: An iterative inorder traversal using an explicit stack visits nodes in ascending order and can stop the instant the k-th node is reached, giving O(h + k) time in the best case and O(n) worst case — optimal because finding the k-th smallest value fundamentally requires examining the k smallest values in order, and inorder traversal is the only traversal that produces a BST's values in sorted order without any extra sorting step.
- Recognize the pattern: Use the list as a practice queue: pick one problem, write the brute-force version first (collect every value via any traversal, sort them, index into position k-1), identify that a BST's inorder traversal is already sorted, then replace the sort with a direct inorder walk that stops early.
- Code walkthrough:
- Defines `kth_smallest(root, k)`: uses an explicit stack to simulate inorder traversal iteratively (avoiding recursion), pushing all left descendants before processing a node.
- Each time a node is popped (visited in ascending order), decrements `k`; when `k` hits 0, that node's value is the k-th smallest and the function returns immediately without visiting the remaining nodes.
- On the BST rooted at `5` with left subtree `3(2, 4)` and right child `8`, the inorder sequence is `2, 3, 4, 5, 8`; the 3rd smallest is `4`, which the function returns.

**Learn more:**
- Website: [GeeksforGeeks: Binary Search Tree](https://www.geeksforgeeks.org/binary-search-tree-data-structure/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Validate+BST+data+structures+algorithms)

**Trade-offs:**
- Stopping early once `k` reaches 0 means this only costs O(h + k) in the best/average case rather than always paying O(n) for a full traversal; a recursive version that collects all values into a list first would always pay O(n) time and O(n) space regardless of k.
- This handles a single query for a fixed `k` in O(h + k); if the BST supports many repeated k-th-smallest queries, augmenting each node with a subtree-size count lets each query run in O(h) with no traversal at all, at the cost of maintaining that count through every insert/delete.

**Practical software engineering use cases:**
- When to use it: Use inorder-based order-statistics for 'k-th smallest/largest' queries, finding a value's rank, or extracting a sorted range from a BST.
- When not to use it: Don't use plain inorder traversal for repeated order-statistic queries at scale — augment the tree with subtree-size counters (an 'order-statistics tree') so each query is O(h) instead of O(h + k).


In [ ]:
# Binary Search Trees - Common problems: Validate BST, Kth Smallest Element in BST, Lowest Common Ancestor of BST
# Kth Smallest Element in BST solved in full below; Validate BST and LCA of BST are already covered earlier in this section.
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def kth_smallest(root, k):
    stack = []
    node = root
    while stack or node:
        while node:
            stack.append(node)
            node = node.left
        node = stack.pop()
        k -= 1
        if k == 0:
            return node.val
        node = node.right
    return None

bst = TreeNode(5, TreeNode(3, TreeNode(2), TreeNode(4)), TreeNode(8))
print(kth_smallest(bst, 3))  # 4, since inorder is 2, 3, 4, 5, 8


In [40]:
# Practice: Binary Search Trees

# Problem:
# Approach:
# Time Complexity:
# Space Complexity:
# Edge Cases:



## Heaps / Priority Queues

### Study checklist
- [ ] Min heap and max heap
- [ ] Top K problems
- [ ] Running median idea
- [ ] Heap-based scheduling problems
- [ ] Common problems: Kth Largest Element, Last Stone Weight, Task Scheduler, Find Median From Data Stream

### Notes
Write your understanding, patterns, mistakes, and edge cases here.


### Beginner-friendly intro
A heap is a tree-shaped structure that always keeps the smallest or largest element at the top.
A priority queue is an interface built on top of a heap where every element has a priority and you always remove the highest (or lowest) priority first.
You use them when you repeatedly need the current best element, like scheduling tasks or computing top K values.


### Checklist item: Min heap and max heap

**Approach:**
- Why this matters: A heap exists because many problems only ever need the current best (min or max) item, not a fully sorted collection; sorting the whole collection every time you need the best item is wasted work that a heap avoids.
- Why this is the optimal approach: A binary heap keeps the smallest element at index 0 by maintaining a partial order (each parent <= its children) rather than a total order, so push and pop are O(log n) — the heap property is cheaper to maintain than full sortedness (O(n log n) to sort) while still answering 'what's the minimum' in O(1) and updating it in O(log n).
- Recognize the pattern: Decide what priority means for your problem (smallest first or largest first), push candidates in as they arrive, and pop when you need the current best — Python's `heapq` is always a min-heap, so negate values if you need max-heap behavior.
- Code walkthrough:
- Pushes `5`, `1`, and `3` onto the heap one at a time; `heapq` enforces the min-heap property after each push.
- Pops and prints `1` — the current minimum, demonstrating that the smallest element is always accessible in O(log n).

**Learn more:**
- Website: [GeeksforGeeks: Heap Data Structure](https://www.geeksforgeeks.org/heap-data-structure/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Min+heap+and+max+heap+data+structures+algorithms)

**Trade-offs:**
- A heap gives O(log n) push/pop and O(1) peek at the minimum, but O(n) to find or remove an arbitrary non-minimum element — if you need fast arbitrary lookups too, you need a heap combined with a hash map (a common pattern for 'indexed priority queue').
- Python's `heapq` module only implements a min-heap directly; simulating a max-heap by negating every value (`-value` on push, negate again on pop) is a standard, well-known workaround rather than a real limitation, but it's easy to forget and produce sign-flipped bugs.

**Practical software engineering use cases:**
- When to use it: Use a heap whenever you repeatedly need 'the current best item' without needing the whole collection sorted — schedulers, Dijkstra's algorithm, or streaming top-k.
- When not to use it: Don't use a heap if you need the full collection in sorted order at once — just call `sorted()`; a heap only pays off when you extract items one at a time or interleave insertions with extractions.


In [41]:
# Python heapq is a min-heap.
import heapq
heap = []
for value in [5, 1, 3]:
    heapq.heappush(heap, value)
print(heapq.heappop(heap))


1


### Checklist item: Top K problems

**Approach:**
- Why this matters: Finding the top K largest values by sorting the entire array first is O(n log n) and wastes effort fully ordering the n-k elements you don't care about; a size-bounded heap tracks only what's needed to answer the question.
- Why this is the optimal approach: Maintaining a min-heap capped at size k, popping the smallest whenever the heap exceeds k, does O(log k) work per element for O(n log k) total time — this is optimal versus a full sort's O(n log n) whenever k is small relative to n, because the heap only ever does comparisons against the current k candidates, never against the other n-k elements once they're discarded.
- Recognize the pattern: Decide what priority means (smallest of the current top-k, since that's the one at risk of being evicted), push each new candidate in, and pop the smallest whenever the heap exceeds size k — the heap always holds exactly the k largest values seen so far.
- Code walkthrough:
- Pushes each value and immediately pops the smallest if the heap exceeds `k = 3` elements.
- After all five numbers, the heap retains the three largest: the popped elements were the two smallest.
- Prints them sorted descending: `[9, 7, 5]`.

**Learn more:**
- Website: [GeeksforGeeks: Heap Data Structure](https://www.geeksforgeeks.org/heap-data-structure/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Top+K+problems+data+structures+algorithms)

**Trade-offs:**
- A min-heap of size k for 'top-k largest' is a slightly counterintuitive but standard trick: you evict the *smallest* of the current top-k candidates, because that's the one most likely to be replaced by a larger incoming value.
- This is O(n log k) time and O(k) space; if k is very close to n, a full sort (O(n log n) time but simpler code) is not meaningfully worse and easier to reason about — the heap's advantage is clearest when k is much smaller than n.

**Practical software engineering use cases:**
- When to use it: Use a size-bounded heap for streaming top-k problems: top-k trending items, k nearest points, or k largest/smallest values from a data stream too large to sort all at once.
- When not to use it: Don't use a size-bounded heap if you need the top-k *in sorted order* as a side effect at every step — popping from this heap gives you the smallest of the current top-k, not a fully ordered top-k list, without an extra sort at the end.


In [42]:
# Top K largest values with a small min-heap.
import heapq
nums = [5, 1, 9, 3, 7]
k = 3
heap = []
for value in nums:
    heapq.heappush(heap, value)
    if len(heap) > k:
        heapq.heappop(heap)
print(sorted(heap, reverse=True))


[9, 7, 5]


### Checklist item: Running median idea

**Approach:**
- Why this matters: Recomputing the median from scratch on every new data point (sort everything, then index the middle) costs O(n log n) per insertion, which is untenable for a streaming median where new values keep arriving; the two-heap technique keeps the median available in O(1) with cheap O(log n) updates.
- Why this is the optimal approach: Splitting values into a max-heap for the lower half and a min-heap for the upper half, kept balanced within one element of each other, lets you read the median in O(1) (from the heap tops) and insert a new value in O(log n) — optimal because a full re-sort per insertion is O(n log n), and no data structure can answer 'what's the median so far' in better than O(1) read once you accept O(log n) insert, since some ordering work must happen incrementally.
- Recognize the pattern: Decide what priority means for each half independently (max-heap for 'largest of the small half', min-heap for 'smallest of the large half'), push new values in and rebalance so the two heaps never differ in size by more than one — the median is then always at one or both heap tops.
- Code walkthrough:
- Maintains two heaps: `low` (a max-heap stored as negatives) for the lower half and `high` (a min-heap) for the upper half.
- Each new value enters `low`, then `low`'s maximum moves to `high`; if `high` grows larger, its minimum moves back to `low`.
- After each insertion the median is either `-low[0]` (odd count) or the average of the two heap tops (even count).

**Learn more:**
- Website: [GeeksforGeeks: Heap Data Structure](https://www.geeksforgeeks.org/heap-data-structure/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Running+median+idea+data+structures+algorithms)

**Trade-offs:**
- Simulating a max-heap with negated values (`heapq.heappush(low, -value)`) doubles the mental overhead of reasoning about sign flips compared to a real max-heap implementation, but Python's standard library only provides a min-heap, so this is the accepted trade-off for using `heapq` directly.
- This gives O(log n) insertion and O(1) median lookup at any point; if you only ever need the median once at the end (not after every insertion), a single O(n log n) sort followed by direct indexing is simpler and avoids maintaining two heaps.

**Practical software engineering use cases:**
- When to use it: Use the two-heap technique for running statistics on a live data stream: streaming median, percentile tracking, or any 'balance point' that must stay current as data arrives.
- When not to use it: Don't maintain two heaps if the median is only needed once on a static, already-collected dataset — sort once and index directly; two heaps only pay off when insertions are ongoing.


In [43]:
# Running median with two heaps.
import heapq
low = []
high = []
for value in [5, 15, 1, 3]:
    heapq.heappush(low, -value)
    heapq.heappush(high, -heapq.heappop(low))
    if len(high) > len(low):
        heapq.heappush(low, -heapq.heappop(high))
    median = -low[0] if len(low) > len(high) else (-low[0] + high[0]) / 2
    print(median)


5
10.0
5
4.0


### Checklist item: Heap-based scheduling problems

**Approach:**
- Why this matters: Scheduling by priority (earliest finish time, highest urgency, lowest cost) is a different problem than scheduling by arrival order (which a plain queue handles); a heap is what lets 'the next task to run' be recomputed in O(log n) every time priorities change or new tasks arrive.
- Why this is the optimal approach: Building the initial heap with `heapify` is O(n) (not O(n log n), because heapify uses a bottom-up sift that does less total work than n individual pushes), and each subsequent pop is O(log n) — optimal because you need at least O(n) to examine every task once, and heapify achieves that exactly while still enabling O(log n) future pops.
- Recognize the pattern: Decide what priority means (here, earliest finish time as the tuple's first element, so natural tuple comparison sorts by it automatically), heapify the initial task list once, then pop the minimum whenever you need to know 'what runs next'.
- Code walkthrough:
- Converts the task list to a min-heap in O(n) with `heapify`, using each tuple's first element (finish time) as the priority.
- Pops tasks in ascending finish-time order: `(1, 'lint')`, `(2, 'test')`, `(3, 'build')`.

**Learn more:**
- Website: [GeeksforGeeks: Heap Data Structure](https://www.geeksforgeeks.org/heap-data-structure/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Heap-based+scheduling+problems+data+structures+algorithms)

**Trade-offs:**
- `heapq.heapify` is O(n), strictly cheaper than pushing n items one at a time (O(n log n)); always prefer heapify when you have the full initial task list upfront rather than looping with individual `heappush` calls.
- Using a plain tuple `(finish_time, name)` for priority is simple, but if two tasks share the same finish time, Python will compare `name` next as a tiebreaker — for non-comparable payloads (like dicts), you'd need to add an explicit tiebreaker (e.g., an insertion counter) to avoid a `TypeError`.

**Practical software engineering use cases:**
- When to use it: Use heap-based scheduling for job schedulers, event simulation (process events in timestamp order), or any system that must always run 'the next most urgent thing' as priorities or arrivals change.
- When not to use it: Don't use a heap for scheduling when all tasks share the same priority and only arrival order matters — a plain FIFO queue is simpler and sufficient there.


In [44]:
# Schedule by earliest finish time.
import heapq
tasks = [(3, "build"), (1, "lint"), (2, "test")]
heapq.heapify(tasks)
while tasks:
    finish_time, name = heapq.heappop(tasks)
    print(finish_time, name)


1 lint
2 test
3 build


### Checklist item: Common problems: Kth Largest Element, Last Stone Weight, Task Scheduler, Find Median From Data Stream

**Approach:**
- Why this matters: This item exists to turn heap pattern-recognition into a working solution on a genuinely different problem (cooldown-constrained scheduling) than the top-k and running-median items already covered, so combining a heap with an explicit cooldown queue gets real practice.
- Why this is the optimal approach: Using a max-heap of remaining task counts plus a cooldown queue processes each unit of time in O(log T) where T is the number of distinct task types, and the total runtime is bounded by the number of CPU intervals actually needed — this is optimal because scheduling the most frequent remaining task first is a greedy choice that provably minimizes total idle time: delaying the most frequent task only risks running out of other tasks to fill the cooldown gap, never helps.
- Recognize the pattern: Use the list as a practice queue: pick one problem, write the brute-force version first (simulate greedily picking any valid non-cooldown task each tick with a linear scan), identify that a heap gives O(log T) access to 'most frequent remaining task' instead of a linear scan, then replace the scan with heap operations.
- Code walkthrough:
- Defines `least_interval(tasks, n)`: counts each task type's frequency, then pushes negated counts onto a max-heap (via `heapq`, which is min-heap-only, so counts are negated).
- Each simulated time tick pops the most frequent remaining task, decrements its count, and — if it still has remaining instances — parks it in a `cooldown` list tagged with the tick at which it becomes eligible again (`time + n`).
- On `['A','A','A','B','B','B']` with cooldown `n=2`, the answer is `8`: the optimal schedule is `A B idle A B idle A B`, where two idle slots are unavoidable because A and B each need 2 full cooldown gaps between their 3 uses.

**Learn more:**
- Website: [GeeksforGeeks: Heap Data Structure](https://www.geeksforgeeks.org/heap-data-structure/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Kth+Largest+Element+data+structures+algorithms)

**Trade-offs:**
- This greedy heap-based approach runs in time proportional to the schedule length rather than needing to explore alternative orderings; a naive backtracking search over all valid task orderings would be exponential and is never the right choice once the greedy proof (most-frequent-first is always safe) is understood.
- The `cooldown` list here is checked and popped with `cooldown[0]`, which works because tasks are always appended in increasing eligibility-time order; if tasks could become eligible out of order, a second heap (keyed on eligibility time) would be needed instead of a plain list.

**Practical software engineering use cases:**
- When to use it: Use heap-plus-cooldown scheduling for CPU task scheduling with enforced gaps, rate-limited job dispatch, or any 'run the most urgent thing, but respect a mandatory cooldown' constraint.
- When not to use it: Don't use this pattern when there's no cooldown constraint at all — a plain max-heap (without the extra cooldown bookkeeping) is simpler and sufficient for unconstrained priority scheduling.


In [ ]:
# Heaps / Priority Queues - Common problems: Kth Largest Element, Last Stone Weight, Task Scheduler, Find Median From Data Stream
# Task Scheduler solved in full below; the remaining problems stay on the practice queue.
import heapq
from collections import Counter

def least_interval(tasks, n):
    counts = Counter(tasks)
    heap = [-count for count in counts.values()]
    heapq.heapify(heap)
    time = 0
    cooldown = []
    while heap or cooldown:
        time += 1
        if heap:
            count = heapq.heappop(heap) + 1
            if count < 0:
                cooldown.append((count, time + n))
        if cooldown and cooldown[0][1] == time:
            heapq.heappush(heap, cooldown.pop(0)[0])
    return time

print(least_interval(["A", "A", "A", "B", "B", "B"], 2))  # 8

practice_queue = [
    {'problem': 'Kth Largest Element', 'topic': 'Heaps / Priority Queues', 'status': 'todo'},
    {'problem': 'Last Stone Weight', 'topic': 'Heaps / Priority Queues', 'status': 'todo'},
    {'problem': 'Find Median From Data Stream', 'topic': 'Heaps / Priority Queues', 'status': 'todo'}
]

for entry in practice_queue:
    print(f"{entry['topic']}: {entry['problem']} -> {entry['status']}")


In [46]:
# Practice: Heaps / Priority Queues

# Problem:
# Approach:
# Time Complexity:
# Space Complexity:
# Edge Cases:



## Graphs

### Study checklist
- [ ] Adjacency list and adjacency matrix
- [ ] Directed vs undirected graphs
- [ ] Visited set
- [ ] Connected components
- [ ] Cycle detection
- [ ] Common problems: Number of Islands, Clone Graph, Course Schedule, Pacific Atlantic Water Flow

### Notes
Write your understanding, patterns, mistakes, and edge cases here.


### Beginner-friendly intro
A graph is a set of nodes connected by edges that can represent relationships like links, roads, or dependencies.
Graphs can be directed (one-way arrows) or undirected (two-way connections), and may have weights (costs) on edges.
Graph questions are usually about exploring reachability, connectivity, paths, or cycles among nodes.


### Checklist item: Adjacency list and adjacency matrix

**Approach:**
- Why this matters: Choosing between an adjacency list and an adjacency matrix isn't cosmetic — it determines whether checking 'are these two nodes connected' is O(1) (matrix) or O(degree) (list), and whether iterating all edges of a sparse graph costs O(V^2) (matrix) or O(V+E) (list), so picking wrong for the graph's density directly costs performance.
- Why this is the optimal approach: An adjacency list uses O(V + E) space and lets you iterate a node's neighbors in O(degree) time, which is optimal for sparse graphs (E << V^2) because an adjacency matrix would waste O(V^2) space storing mostly-absent edges — the right representation is the one whose complexity matches the graph's actual density, not a universal default.
- Recognize the pattern: Choose adjacency-list representation for sparse graphs (most real-world graphs) and adjacency-matrix for dense graphs or when O(1) edge-existence checks matter more than memory; initialize an empty neighbor list per node with `setdefault` so nodes with no explicit entry still behave correctly.
- Code walkthrough:
- Iterates over the edge list, using `setdefault` to initialise an empty neighbour list for any new node.
- Adds both directions for each undirected edge, so `adj['A']` ends up as `['B', 'C']` and `adj['B']` as `['A', 'C']`.

**Learn more:**
- Website: [GeeksforGeeks: Graph Algorithms](https://www.geeksforgeeks.org/graph-data-structure-and-algorithms/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Adjacency+list+and+adjacency+matrix+data+structures+algorithms)

**Trade-offs:**
- An adjacency list is memory-efficient for sparse graphs but answers 'is there an edge between A and B' in O(degree(A)) time (must scan A's neighbor list); an adjacency matrix answers the same question in O(1) but costs O(V^2) memory regardless of how few edges actually exist.
- This builds an undirected adjacency list by adding both directions per edge; for a directed graph, only add `adj[a].append(b)` — adding both directions when the graph is actually directed silently turns every one-way relationship into a two-way one.

**Practical software engineering use cases:**
- When to use it: Use an adjacency list for most real-world graphs (social networks, road networks, dependency graphs), which are typically sparse (E is much less than V^2).
- When not to use it: Don't use an adjacency list when you need frequent O(1) edge-existence checks on a dense or small graph — an adjacency matrix trades memory for that guarantee.


In [47]:
# Adjacency list vs matrix.
edges = [("A", "B"), ("A", "C"), ("B", "C")]
adj = {}
for a, b in edges:
    adj.setdefault(a, []).append(b)
    adj.setdefault(b, []).append(a)
print(adj)


{'A': ['B', 'C'], 'B': ['A', 'C'], 'C': ['A', 'B']}


### Checklist item: Directed vs undirected graphs

**Approach:**
- Why this matters: Whether a graph is directed changes the meaning of every traversal and algorithm run on it — a road with a one-way street modeled as undirected would let a traversal algorithm 'discover' a path that doesn't actually exist, silently producing wrong answers in routing, dependency-ordering, or reachability code.
- Why this is the optimal approach: Adding an edge in one direction is O(1); adding it in both directions for undirected edges is also O(1), so representing directedness costs nothing extra in time — the correctness win instead comes from matching the model to reality, since the *complexity class* of algorithms like cycle detection or topological sort depends on whether edges are directed, not on the O(1) cost of adding them.
- Recognize the pattern: Choose adjacency representation, then explicitly decide per edge (or per graph) whether to add one direction (directed) or both (undirected) — never assume; the `directed` flag should be as visible and deliberate as the edge data itself.
- Code walkthrough:
- `add_edge` always records `a → b`; it only adds `b → a` when `directed=False`.
- After one directed edge `A→B` and one undirected edge `B↔C`, the graph shows `'A'` pointing only to `'B'` while `'B'` and `'C'` are mutually linked.

**Learn more:**
- Website: [GeeksforGeeks: Graph Algorithms](https://www.geeksforgeeks.org/graph-data-structure-and-algorithms/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Directed+vs+undirected+graphs+data+structures+algorithms)

**Trade-offs:**
- Directed graphs can model asymmetric relationships (one-way dependencies, follower/following) that undirected graphs cannot; but algorithms that assume undirected symmetry (like naive connected-components counting) give wrong answers if silently run on a directed graph without adaptation.
- This mixes a directed and an undirected edge in one graph via a per-call flag, which is flexible for teaching but risky in production — mixing edge types in one graph object without a clear invariant is a common source of bugs; most production graph libraries pick one edge semantics per graph object.

**Practical software engineering use cases:**
- When to use it: Use directed edges for dependencies, workflows, follower relationships, or anything with inherent one-way asymmetry; use undirected edges for symmetric relationships like friendships or physical connections.
- When not to use it: Don't default to undirected 'just in case' — if the real-world relationship is actually one-directional (like a prerequisite), modeling it as undirected breaks algorithms like topological sort that depend on directionality.


In [48]:
# Directed edges add one direction; undirected edges add both.
def add_edge(graph, a, b, directed=False):
    graph.setdefault(a, []).append(b)
    if not directed:
        graph.setdefault(b, []).append(a)

graph = {}
add_edge(graph, "A", "B", directed=True)
add_edge(graph, "B", "C", directed=False)
print(graph)


{'A': ['B'], 'B': ['C'], 'C': ['B']}


### Checklist item: Visited set

**Approach:**
- Why this matters: Without a visited set, DFS/BFS on any graph containing a cycle recurses or loops forever, because nothing stops the traversal from walking back to a node it already processed — this single omission is the most common cause of infinite loops in graph code.
- Why this is the optimal approach: Checking and updating a hash set in O(1) average time per node means the visited-guard adds no asymptotic cost to the traversal — the overall traversal remains O(V + E), which is optimal because you must at least look at every reachable vertex and edge once, and a hash-set check is the cheapest possible way to avoid revisiting a vertex.
- Recognize the pattern: Choose adjacency representation, initialize an empty visited set before traversal begins, and check-and-add to it at the very start of processing each node — checking membership before recursing (not after) is what actually prevents infinite recursion on a cycle.
- Code walkthrough:
- Builds a cyclic graph `{1:[2], 2:[3], 3:[1]}`, which would loop forever without a visited guard.
- `dfs` returns immediately when `node in seen`, preventing infinite recursion on the cycle.
- Prints `{1, 2, 3}` — all three nodes visited exactly once.

**Learn more:**
- Website: [GeeksforGeeks: Graph Algorithms](https://www.geeksforgeeks.org/graph-data-structure-and-algorithms/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Visited+set+data+structures+algorithms)

**Trade-offs:**
- A set gives O(1) average membership checks; for graphs where nodes are already dense small integers (0 to n-1), a boolean array is faster and uses less memory than a hash set, though a set is more flexible for arbitrary hashable node labels.
- This particular `dfs` checks `if node in seen: return` at the top, which means the same node can technically be pushed onto the call stack multiple times before the check fires; an alternative that checks *before* recursing into a neighbor (rather than at the top of the callee) avoids that redundant call entirely.

**Practical software engineering use cases:**
- When to use it: Always use a visited set for any traversal on a graph that might contain cycles or that you haven't proven is acyclic — the cost of the check is negligible compared to the cost of an infinite loop.
- When not to use it: You can skip the visited set only when the structure is provably acyclic with no shared ancestors (a tree) — adding it there is harmless but unnecessary bookkeeping.


In [49]:
# A visited set prevents repeated work and infinite loops.
graph = {1: [2], 2: [3], 3: [1]}
seen = set()
def dfs(node):
    if node in seen:
        return
    seen.add(node)
    for neighbor in graph[node]:
        dfs(neighbor)
dfs(1)
print(seen)


{1, 2, 3}


### Checklist item: Connected components

**Approach:**
- Why this matters: Counting connected components answers 'how many separate groups exist' — a question that shows up directly in clustering, network-partition detection, and 'is everything reachable from here' checks, and it's a direct, practical payoff of combining traversal with a visited set correctly.
- Why this is the optimal approach: Running a traversal from every unvisited node and reusing one shared `seen` set across all of them visits each vertex and edge exactly once in total (not once per component attempt), giving O(V + E) time overall — optimal because you cannot determine which component a vertex belongs to without visiting it, and sharing the visited set across components is what prevents the naive mistake of re-traversing the whole graph for every node.
- Recognize the pattern: Choose adjacency representation, initialize one visited set for the whole graph (not per-component), and traverse from each still-unvisited node — every node reachable from that traversal belongs to the same component, so the outer loop only starts a new count when it hits a genuinely fresh, unvisited node.
- Code walkthrough:
- For each unvisited node, increments `components` and calls `dfs` to mark every reachable node as seen.
- Uses the same `seen` set across all DFS calls so nodes visited in one component are skipped by later outer-loop iterations.
- Prints `3` — groups `{0,1}`, `{2,3}`, and `{4}` are the three components.

**Learn more:**
- Website: [GeeksforGeeks: Graph Algorithms](https://www.geeksforgeeks.org/graph-data-structure-and-algorithms/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Connected+components+data+structures+algorithms)

**Trade-offs:**
- Using one shared `seen` set across all DFS calls (rather than a fresh one per component) is what keeps this O(V + E) instead of O(V * (V + E)) — reusing state across the outer loop is the entire optimization, not an implementation detail.
- This counts components but doesn't return their membership; if you need to know *which* nodes belong to which component (not just the count), collect each `dfs` call's visited nodes into a group before moving to the next unvisited node.

**Practical software engineering use cases:**
- When to use it: Use connected-components counting for network partition detection, clustering related records without explicit IDs, or verifying that a graph is fully connected (exactly one component).
- When not to use it: Don't use a fresh traversal per node without sharing visited state — that's the classic O(V^2) mistake that defeats the purpose of using a linear-time traversal in the first place.


In [50]:
# Count connected components.
graph = {0: [1], 1: [0], 2: [3], 3: [2], 4: []}
seen = set()
def dfs(node):
    seen.add(node)
    for neighbor in graph[node]:
        if neighbor not in seen:
            dfs(neighbor)
components = 0
for node in graph:
    if node not in seen:
        components += 1
        dfs(node)
print(components)


3


### Checklist item: Cycle detection

**Approach:**
- Why this matters: Detecting a cycle in a directed graph is not the same problem as an undirected graph's visited-set check — a directed graph can revisit an already-fully-processed node through a different path with no cycle at all, so a plain visited set alone would produce false positives; you need to distinguish 'currently on my path' from 'finished and safe'.
- Why this is the optimal approach: Three-coloring each node (white/gray/black) and checking specifically for an edge to a gray (in-progress) node does O(1) extra work per edge, keeping the whole detection at O(V + E) — optimal because a directed cycle exists if and only if DFS ever finds a back edge to a node still on the current recursion stack, and three-coloring is the minimal state needed to distinguish that case from a safe cross-edge to an already-finished node.
- Recognize the pattern: Color each node unvisited (white) / in-progress on the current DFS path (gray) / fully finished (black); a back edge to a gray node reveals a cycle, while an edge to a black node is always safe because that subtree already finished without looping back.
- Code walkthrough:
- Colours nodes `'white'` (unvisited), `'gray'` (on the current DFS path), or `'black'` (fully processed).
- A back edge to a `'gray'` ancestor signals a directed cycle; the function returns `True` immediately.
- Prints `True` for the graph `A→B→C→A`, which contains a directed cycle.

**Learn more:**
- Website: [GeeksforGeeks: Graph Algorithms](https://www.geeksforgeeks.org/graph-data-structure-and-algorithms/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Cycle+detection+data+structures+algorithms)

**Trade-offs:**
- Using three colors instead of a single visited boolean is what correctly distinguishes 'this node is an ancestor on my current path' (gray, a real cycle if revisited) from 'this node was already fully explored via a different path' (black, safe) — collapsing this to a single boolean would incorrectly flag every diamond-shaped DAG (two paths converging on one node) as a cycle.
- This DFS-coloring approach detects a cycle but doesn't extract the actual cycle's nodes; if you need to report which nodes form the cycle, you must additionally track the path (e.g., a parent-pointer stack) so you can walk back from the gray node that was re-encountered.

**Practical software engineering use cases:**
- When to use it: Use three-color DFS cycle detection for validating dependency graphs (build systems, package managers), detecting deadlock-prone wait-for graphs, or confirming a workflow graph is a valid DAG before running topological sort on it.
- When not to use it: Don't use this three-color scheme on an undirected graph — undirected cycle detection only needs a visited set plus tracking the parent edge (to avoid falsely flagging the edge you just came from as a cycle).


In [51]:
# Detect a cycle in a directed graph with DFS colors.
graph = {"A": ["B"], "B": ["C"], "C": ["A"]}
color = {node: "white" for node in graph}
def has_cycle(node):
    color[node] = "gray"
    for neighbor in graph[node]:
        if color.get(neighbor) == "gray":
            return True
        if color.get(neighbor, "white") == "white" and has_cycle(neighbor):
            return True
    color[node] = "black"
    return False
print(any(has_cycle(node) for node in graph if color[node] == "white"))


True


### Checklist item: Common problems: Number of Islands, Clone Graph, Course Schedule, Pacific Atlantic Water Flow

**Approach:**
- Why this matters: This item exists to turn graph pattern-recognition into a working solution on a genuinely different problem (implicit grid graphs) than the explicit adjacency-list items already covered, so recognizing that a 2D grid is itself a graph — where cells are nodes and adjacency is 'neighboring cell' — gets real practice.
- Why this is the optimal approach: Treating each land cell as a node and flood-filling connected land with DFS/BFS visits each of the R*C cells and each of its O(1) grid-neighbors exactly once, giving O(R*C) time and O(R*C) space for the visited set — optimal because you must inspect every cell at least once to know whether it's land or water, and this algorithm does exactly that with no cell revisited.
- Recognize the pattern: Use the list as a practice queue: pick one problem, write the brute-force version first (for every land cell, re-scan the whole grid to see what it's connected to), identify that a flood-fill from each unvisited land cell already discovers its whole island in one pass, then replace repeated re-scans with a single shared visited set (exactly the connected-components pattern from earlier, applied to an implicit grid graph instead of an explicit adjacency list).
- Code walkthrough:
- Defines `num_islands(grid)`: treats the 2D grid as an implicit graph where each `'1'` cell is a node and its up/down/left/right neighbors (if also `'1'`) are edges — no explicit adjacency list is ever built.
- `dfs(r, c)` uses an explicit stack (avoiding recursion depth limits on large grids) to flood-fill every land cell reachable from `(r, c)`, marking each as seen the moment it's discovered.
- The outer double loop increments `islands` only when it finds a `'1'` cell not yet in `seen`, then flood-fills the rest of that island — on the sample 4x5 grid this correctly counts 3 separate islands.

**Learn more:**
- Website: [GeeksforGeeks: Graph Algorithms](https://www.geeksforgeeks.org/graph-data-structure-and-algorithms/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Number+of+Islands+data+structures+algorithms)

**Trade-offs:**
- Using an explicit stack for the flood-fill avoids Python's recursion depth limit on large grids (a purely recursive DFS could hit `RecursionError` on a grid with a long snaking island); the trade-off is slightly more verbose code than a two-line recursive DFS.
- This mutates no input data (uses a separate `seen` set); an alternative in-place approach that overwrites visited land cells directly in the grid (e.g., setting them to `'0'`) saves the O(R*C) `seen` set memory but destroys the original grid, which matters if the caller needs it afterward.

**Practical software engineering use cases:**
- When to use it: Use grid flood-fill for connected-region counting on 2D data: island counting, connected pixel regions in image processing, or flood-fill paint-bucket tools.
- When not to use it: Don't build an explicit adjacency list for grid problems — treating the grid itself as an implicit graph (computing neighbors on the fly from row/col offsets) avoids the O(R*C) memory and setup cost of materializing edges that are already implied by grid geometry.


In [ ]:
# Graphs - Common problems: Number of Islands, Clone Graph, Course Schedule, Pacific Atlantic Water Flow
# Number of Islands solved in full below; the remaining problems stay on the practice queue.
def num_islands(grid):
    rows, cols = len(grid), len(grid[0])
    seen = set()

    def dfs(r, c):
        stack = [(r, c)]
        while stack:
            row, col = stack.pop()
            for dr, dc in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
                nr, nc = row + dr, col + dc
                if (0 <= nr < rows and 0 <= nc < cols
                        and grid[nr][nc] == "1" and (nr, nc) not in seen):
                    seen.add((nr, nc))
                    stack.append((nr, nc))

    islands = 0
    for r in range(rows):
        for c in range(cols):
            if grid[r][c] == "1" and (r, c) not in seen:
                islands += 1
                seen.add((r, c))
                dfs(r, c)
    return islands

grid = [
    list("11000"),
    list("11000"),
    list("00100"),
    list("00011"),
]
print(num_islands(grid))  # 3

practice_queue = [
    {'problem': 'Clone Graph', 'topic': 'Graphs', 'status': 'todo'},
    {'problem': 'Course Schedule', 'topic': 'Graphs', 'status': 'todo'},
    {'problem': 'Pacific Atlantic Water Flow', 'topic': 'Graphs', 'status': 'todo'}
]

for entry in practice_queue:
    print(f"{entry['topic']}: {entry['problem']} -> {entry['status']}")


In [53]:
# Practice: Graphs

# Problem:
# Approach:
# Time Complexity:
# Space Complexity:
# Edge Cases:



## Advanced Graph + DP

### Study checklist
- [ ] Multi-source BFS on grids and graphs
- [ ] DP on DAGs (longest/shortest path via topological order)
- [ ] Minimum spanning tree basics: Prim vs Kruskal and when to use each
- [ ] Using Union-Find inside Kruskal and complexity trade-offs
- [ ] Strongly connected components (SCC) at a high level
- [ ] Max-flow / min-cut as an optional stretch concept
- [ ] Bipartite matching as an optional stretch concept

### Notes
Write your understanding, trade-offs, and typical system-style use-cases here (capacity planning, routing, dependency analysis).

### Beginner-friendly intro
This section combines graph traversal with optimization to solve problems that plain BFS or DFS cannot handle efficiently: connecting the graph at minimum cost, pushing maximum flow, finding strongly connected regions, or computing optimal paths on directed graphs.
You reach for these techniques when a problem adds weights to edges, enforces direction, or requires a globally optimal structure rather than just any valid path.
Questions here typically ask for the cheapest spanning tree, the longest or most-counted path through a dependency graph, the maximum throughput through a network, or the identification of tightly linked cycles in a directed graph.

### Checklist item: Multi-source BFS on grids and graphs

**Approach:**
- Why this matters: Running a separate single-source BFS from every source and taking the minimum distance per cell is correct but wasteful — it repeats work, since a cell equidistant from two sources gets computed twice; multi-source BFS gets the same 'distance to nearest source' answer in a single pass.
- Why this is the optimal approach: Seeding the queue with all sources simultaneously at distance 0 and expanding layer by layer visits each cell exactly once, giving O(R*C) time for an R-by-C grid — optimal because computing every cell's distance to its nearest source requires examining every cell at least once, and running k separate BFS passes (one per source) would cost O(k*R*C) instead, strictly more work for the identical answer.
- Recognize the pattern: Put every starting state into the queue at once (all at distance 0), mark them all seen up front, then process one distance-layer at a time — the key insight is that 'nearest source' falls out for free when all sources start the race simultaneously, rather than needing to compare k separate single-source results afterward.
- Code walkthrough:
- Seeds the BFS queue with all source cells (value `1`) simultaneously at distance `0`, and marks their `dist` entries.
- Expands outward layer by layer; each newly reached cell gets `dist[r][c] + 1`, so every cell ends up with its distance to the nearest source.
- Prints the completed `dist` grid — cells that were already sources show `0`; all others show the hop count to the nearest `1`.

**Learn more:**
- Website: [cp-algorithms: Breadth First Search](https://cp-algorithms.com/graph/breadth-first-search.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Multi-source+BFS+on+grids+and+graphs+data+structures+algorithms)

**Trade-offs:**
- Multi-source BFS computes 'distance to the single nearest source' for every cell in one O(R*C) pass; if you instead need the distance from every cell to *every* source individually, you're back to needing k separate BFS runs — multi-source BFS only collapses the work when the minimum-over-sources is what you actually need.
- This assumes uniform edge weight (each grid step costs 1); if some cells are more costly to enter than others (weighted grid), plain BFS no longer guarantees shortest distances and you need Dijkstra's algorithm with a priority queue instead.

**Practical software engineering use cases:**
- When to use it: Use multi-source BFS for 'distance to nearest X' problems: nearest hospital/exit on a map, rotting-oranges-style infection spread, or nearest safe cell from multiple hazards.
- When not to use it: Don't use multi-source BFS when you need each source's distance tracked separately rather than the minimum across all sources — run individual BFS passes instead.


In [54]:
# Multi-source BFS starts all sources at distance 0.
from collections import deque
grid = [[0, 1, 0], [0, 0, 0], [1, 0, 0]]
rows, cols = len(grid), len(grid[0])
dist = [[None] * cols for _ in range(rows)]
queue = deque()
for r in range(rows):
    for c in range(cols):
        if grid[r][c] == 1:
            dist[r][c] = 0
            queue.append((r, c))
while queue:
    r, c = queue.popleft()
    for dr, dc in [(1,0), (-1,0), (0,1), (0,-1)]:
        nr, nc = r + dr, c + dc
        if 0 <= nr < rows and 0 <= nc < cols and dist[nr][nc] is None:
            dist[nr][nc] = dist[r][c] + 1
            queue.append((nr, nc))
print(dist)


[[1, 0, 1], [1, 1, 2], [0, 1, 2]]


### Checklist item: DP on DAGs (longest/shortest path via topological order)

**Approach:**
- Why this matters: Computing the longest or shortest path in a general graph with cycles is NP-hard for longest path and needs Dijkstra/Bellman-Ford machinery for shortest path; a DAG's acyclic structure is what makes both solvable in linear time by processing nodes in an order where every predecessor is already finalized before you need it.
- Why this is the optimal approach: Processing nodes in reverse topological order and computing `dp[node] = 1 + max(dp[neighbor] for neighbor in graph[node])` does O(1) amortized work per edge, giving O(V + E) total time — optimal because you must examine every edge at least once to know its contribution to some path, and topological order guarantees each node's dp value is fully computed (all its outgoing paths resolved) before any earlier node needs to read it, so no node is ever revisited.
- Recognize the pattern: Process nodes in topological order (here, in reverse, since we're computing longest path *from* each node looking forward); for each node, compute its value by combining its already-solved successors' values — this only works because a DAG guarantees such an order exists at all.
- Code walkthrough:
- Iterates through nodes in reverse topological order (from `'D'` back to `'A'`).
- For each node with outgoing edges, sets `dp[node] = 1 + max(dp[neighbor])` — the longest path starting at that node.
- Prints `{'A': 2, 'B': 1, 'C': 1, 'D': 0}`: from `A`, the longest path has 2 hops.

**Learn more:**
- Website: [cp-algorithms: Topological Sorting](https://cp-algorithms.com/graph/topological-sort.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=DP+on+DAGs+%28longest%2Fshortest+path+via+topological+order%29+data+structures+algorithms)

**Trade-offs:**
- This DP-on-DAG approach is O(V + E), strictly better than a general-graph shortest/longest-path algorithm (Bellman-Ford is O(V*E)) precisely because it exploits acyclicity; running this same logic on a graph with a cycle would either loop forever or require detecting and rejecting the cycle first.
- This computes longest path by taking `max` at each step; swapping `max` for `min` (with dp initialized to infinity instead of 0, except at the source) computes shortest path instead — the algorithmic shape is identical, only the combining function and base case change.

**Practical software engineering use cases:**
- When to use it: Use topological-order DP for scheduling problems (longest chain of dependent tasks, critical path method), counting paths in a DAG, or any acyclic dependency graph where you need an optimal path or count.
- When not to use it: Don't use this technique if the graph might contain cycles — you must first prove (or enforce via cycle detection) that it's a DAG, or fall back to Bellman-Ford/other general shortest-path algorithms.


In [55]:
# Longest path in a DAG using topological order.
graph = {"A": ["B", "C"], "B": ["D"], "C": ["D"], "D": []}
order = ["A", "B", "C", "D"]
dp = {node: 0 for node in graph}
for node in reversed(order):
    if graph[node]:
        dp[node] = 1 + max(dp[neighbor] for neighbor in graph[node])
print(dp)


{'A': 2, 'B': 1, 'C': 1, 'D': 0}


### Checklist item: Minimum spanning tree basics: Prim vs Kruskal and when to use each

**Approach:**
- Why this matters: Connecting all nodes in a network at minimum total cost (laying cable, wiring circuits, road networks) is a different question than shortest-path-between-two-nodes; naively trying every possible subset of edges to find the minimum spanning tree is exponential, while both Kruskal's and Prim's algorithms solve it greedily in polynomial time.
- Why this is the optimal approach: Kruskal's algorithm sorts all edges once (O(E log E)) and greedily adds each edge that connects two different components (checked via Union-Find in near-O(1) amortized time), for O(E log E) total time; this is optimal because the greedy choice is provably safe — the cut property guarantees that the cheapest edge crossing any cut of the graph belongs to some MST, so always taking the next-cheapest valid edge can never be wrong.
- Recognize the pattern: Sort edges by weight, then greedily add the cheapest edge whose two endpoints are currently in different Union-Find components — skip any edge that would connect two nodes already in the same component, since that edge would only create a cycle, not extend the tree.
- Code walkthrough:
- Sorts edges by weight so the cheapest candidate is always processed first (Kruskal's greedy rule).
- Uses a `find` function with path halving to locate each node's root, then unites two roots only when they belong to different components.
- Appends only the edges that connect two previously separate components; the resulting `mst` is the minimum spanning tree.

**Learn more:**
- Website: [cp-algorithms: Kruskal's MST Algorithm](https://cp-algorithms.com/graph/mst_kruskal.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Minimum+spanning+tree+basics+data+structures+algorithms)

**Trade-offs:**
- Kruskal's algorithm (edge-centric, sort all edges) is typically preferred for sparse graphs since its O(E log E) cost is dominated by the number of edges; Prim's algorithm (vertex-centric, grow one tree using a priority queue of frontier edges) runs in O(E log V) with a binary heap and is often preferred for dense graphs where E is close to V^2.
- Both algorithms find *a* minimum spanning tree, but if edge weights contain ties, different valid MSTs (same total weight, different edge sets) can result depending on tie-breaking order — if you need a canonical/deterministic MST, you must define an explicit tie-breaking rule.

**Practical software engineering use cases:**
- When to use it: Use Kruskal's algorithm (with Union-Find) for MST problems on sparse graphs, or when you already have a sorted or easily-sortable edge list — network design, clustering (via MST edge cuts), or approximate TSP tours.
- When not to use it: Don't use Kruskal's on a dense graph where sorting E ~ V^2 edges dominates the cost — Prim's algorithm with a priority queue is typically faster there since it never needs a full edge sort.


In [56]:
# Kruskal MST sketch with Union-Find.
edges = [(1, "A", "B"), (3, "A", "C"), (2, "B", "C")]
parent = {node: node for edge in edges for node in edge[1:]}
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
mst = []
for weight, a, b in sorted(edges):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[ra] = rb
        mst.append((weight, a, b))
print(mst)


[(1, 'A', 'B'), (2, 'B', 'C')]


### Checklist item: Using Union-Find inside Kruskal and complexity trade-offs

**Approach:**
- Why this matters: Kruskal's algorithm needs to answer 'are these two nodes already connected' up to E times as it processes edges; without an efficient way to answer that, checking connectivity naively (a graph traversal per edge) would cost O(E*(V+E)) instead of the near-linear time Union-Find provides.
- Why this is the optimal approach: Path compression (flattening the tree during find) combined with union by size (always attaching the smaller tree under the larger) gives amortized O(alpha(n)) time per operation, where alpha is the inverse Ackermann function — effectively constant for any n that could exist in practice — making Kruskal's overall O(E log E) time dominated entirely by the initial edge sort, not by the connectivity checks.
- Recognize the pattern: Represent each component by a parent pointer; compress paths during find so future finds on the same nodes are faster, and attach the smaller tree under the larger during union so no tree grows disproportionately tall.
- Code walkthrough:
- Implements full Union-Find: `find` with recursive path compression and `union` with size-based attachment.
- `union(0, 1)` and `union(3, 4)` merge those pairs; `union` returns `False` if both nodes already share a root.
- Prints `True` — `find(1) == find(0)` after they were merged into the same component.

**Learn more:**
- Website: [cp-algorithms: Disjoint Set Union](https://cp-algorithms.com/data_structures/disjoint_set_union.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Using+Union-Find+inside+Kruskal+and+complexity+trade-offs+data+structures+algorithms)

**Trade-offs:**
- Path compression alone (without union by size/rank) still gives good amortized performance (O(log n) per op), but combining both optimizations together is what achieves the near-constant O(alpha(n)) bound — using only one of the two optimizations leaves performance on the table.
- Union-Find answers 'are these connected' and 'merge these two groups' extremely fast, but it cannot efficiently answer 'list all members of this component' or 'remove this edge' — if you need either of those, Union-Find is the wrong structure and you need an adjacency-list-based approach instead.

**Practical software engineering use cases:**
- When to use it: Use Union-Find inside Kruskal's algorithm (or standalone) whenever you need fast, repeated 'are these connected / merge these' queries: MST construction, detecting redundant connections, or incremental connectivity in a dynamic graph.
- When not to use it: Don't use Union-Find when you need to *remove* edges/connections — the structure only supports merging, not splitting, so a dynamic graph with deletions needs a different approach (like a link-cut tree).


In [57]:
# Union-Find with path compression and union by size.
parent = list(range(5))
size = [1] * 5

def find(x):
    if parent[x] != x:
        parent[x] = find(parent[x])
    return parent[x]

def union(a, b):
    ra, rb = find(a), find(b)
    if ra == rb:
        return False
    if size[ra] < size[rb]:
        ra, rb = rb, ra
    parent[rb] = ra
    size[ra] += size[rb]
    return True

union(0, 1)
union(3, 4)
print(find(1) == find(0))


True


### Checklist item: Strongly connected components (SCC) at a high level

**Approach:**
- Why this matters: In a directed graph, 'connected' is ambiguous — two nodes might reach each other in both directions (strongly connected) or only one way; strongly connected components matter because they identify groups of nodes that are mutually reachable, which reveals cyclic dependency clusters and lets you collapse each SCC into a single node for further analysis.
- Why this is the optimal approach: Kosaraju's algorithm runs two DFS passes — one on the original graph to compute a finishing order, one on the transposed (edge-reversed) graph processing nodes in reverse finish order — for O(V + E) total time; this is optimal because identifying SCCs requires examining every edge at least once to determine reachability, and Kosaraju's insight (that processing the transpose graph in reverse-finish order isolates exactly one SCC per DFS tree) achieves that with only two linear passes, no better complexity class being possible.
- Recognize the pattern: Run a first DFS on the original graph, recording each node's finish time (when its recursion fully completes); build the transposed graph (reverse every edge); then run a second DFS on the transpose, processing nodes in decreasing finish-time order — each DFS tree in this second pass is exactly one SCC.
- Code walkthrough:
- **Pass 1:** runs DFS on the original graph, appending each node to `finish_stack` only after all its descendants are processed.
- **Pass 2:** builds the reversed graph `rev`, pops nodes from `finish_stack`, and runs a second DFS on `rev`; each new unvisited node starts a fresh SCC.
- Prints `[['A', 'B', 'C'], ['D']]` — the three-node directed cycle forms one SCC; `D` is isolated because it has no incoming edges from the cycle.

**Learn more:**
- Website: [cp-algorithms: Strongly Connected Components](https://cp-algorithms.com/graph/strongly-connected-components.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Strongly+connected+components+%28SCC%29+at+a+high+level+data+structures+algorithms)

**Trade-offs:**
- Kosaraju's algorithm is conceptually simple (two full DFS passes plus a graph transpose) but requires building the reversed graph explicitly, using O(V + E) extra memory; Tarjan's algorithm finds SCCs in a single DFS pass using a stack and low-link values, avoiding the transpose at the cost of trickier bookkeeping.
- This implementation returns each SCC as a plain list of nodes; if you need the *condensation graph* (each SCC collapsed to a single node, with edges between SCCs), you need an additional pass mapping original edges onto their SCCs' representative nodes.

**Practical software engineering use cases:**
- When to use it: Use SCC detection to find cyclic dependency clusters in build/import graphs, identify mutually-reachable clusters in web-link graphs, or simplify a directed graph by collapsing each SCC before running further analysis (like topological sort on the condensation graph).
- When not to use it: Don't run full SCC detection if you only need to know whether the *whole* graph is strongly connected (a single yes/no) — a lighter check (one DFS from any node, plus one DFS on the transpose from the same node, both reaching all vertices) answers that without computing every individual SCC.


In [ ]:
# Kosaraju's SCC: DFS on original graph records finish order;
# DFS on reversed graph in reverse finish order finds each SCC.
graph = {'A': ['B'], 'B': ['C'], 'C': ['A'], 'D': ['C']}
visited, finish_stack = set(), []

def dfs1(node):
    visited.add(node)
    for nb in graph[node]:
        if nb not in visited:
            dfs1(nb)
    finish_stack.append(node)

for node in graph:
    if node not in visited:
        dfs1(node)

rev = {node: [] for node in graph}
for node, neighbors in graph.items():
    for nb in neighbors:
        rev[nb].append(node)

visited2, sccs = set(), []

def dfs2(node, scc):
    visited2.add(node)
    scc.append(node)
    for nb in rev[node]:
        if nb not in visited2:
            dfs2(nb, scc)

while finish_stack:
    node = finish_stack.pop()
    if node not in visited2:
        scc = []
        dfs2(node, scc)
        sccs.append(sorted(scc))

print('SCCs:', sccs)   # [['A', 'B', 'C'], ['D']]


### Checklist item: Max-flow / min-cut as an optional stretch concept

**Approach:**
- Why this matters: Many real capacity-constrained problems (network bandwidth, pipe flow, bipartite matching itself) reduce to 'what's the maximum flow from a source to a sink given capacity limits on each connection' — a question that greedy single-path routing can't answer correctly because it might use up capacity on a path that blocks a better combination of paths.
- Why this is the optimal approach: The Ford-Fulkerson method repeatedly finds an augmenting path (a path from source to sink with spare capacity) and pushes flow equal to its bottleneck (the minimum remaining capacity along that path) until no augmenting path remains; the Edmonds-Karp refinement specifically uses BFS to find the *shortest* augmenting path each time, guaranteeing termination in O(V*E^2) — this is optimal in the sense that the max-flow min-cut theorem proves the resulting flow is provably maximum: no flow assignment can exceed the capacity of the graph's minimum cut, and Ford-Fulkerson's termination state is exactly when the flow equals that minimum cut's capacity.
- Recognize the pattern: Model the problem as a flow network with capacities on each edge; repeatedly find an augmenting path from source to sink with remaining capacity, determine its bottleneck (the smallest remaining capacity along the path), and subtract that bottleneck from every edge on the path — repeat until no augmenting path exists.
- Code walkthrough:
- Defines edge capacities as a dict and picks one augmenting path `S → A → T` through the network.
- Finds the bottleneck — the minimum remaining capacity along the path — and subtracts it from every edge on the path.
- Prints the flow sent (`2`) and the updated capacities, showing how one augmenting path reduces available bandwidth.

**Learn more:**
- Website: [cp-algorithms: Max Flow (Ford-Fulkerson / Edmonds-Karp)](https://cp-algorithms.com/graph/edmonds_karp.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Max-flow+%2F+min-cut+as+an+optional+stretch+concept+data+structures+algorithms)

**Trade-offs:**
- This sketch picks one hardcoded augmenting path to demonstrate the bottleneck-and-subtract mechanic; a full implementation must search for augmenting paths using BFS (Edmonds-Karp, O(V*E^2)) or DFS (basic Ford-Fulkerson, whose runtime depends on capacity values and can be much slower on graphs with large integer capacities).
- Max-flow algorithms find the maximum total flow value, but if you also need the specific min-cut edges (not just the flow amount), you need an extra step: after the algorithm terminates, the min-cut is exactly the edges from nodes still reachable from the source (in the residual graph) to nodes that are not.

**Practical software engineering use cases:**
- When to use it: Use max-flow/min-cut for network capacity planning, bipartite matching (reducible to a flow problem), image segmentation, or any 'maximum throughput given capacity constraints' question.
- When not to use it: Don't reach for full max-flow machinery when the underlying question is really just bipartite matching on a small graph — Kuhn's augmenting-path algorithm solves that special case more directly without needing to model capacities explicitly.


In [59]:
# Max-flow intuition: each edge has remaining capacity.
capacity = {("S", "A"): 3, ("A", "T"): 2, ("S", "T"): 1}
flow_path = [("S", "A"), ("A", "T")]
bottleneck = min(capacity[edge] for edge in flow_path)
for edge in flow_path:
    capacity[edge] -= bottleneck
print("sent flow:", bottleneck)
print(capacity)


sent flow: 2
{('S', 'A'): 1, ('A', 'T'): 0, ('S', 'T'): 1}


### Checklist item: Bipartite matching as an optional stretch concept

**Approach:**
- Why this matters: Greedily matching each left node to its first available preferred right node (as in the sample code) can get stuck in a locally-good-but-globally-suboptimal matching — the whole point of a proper bipartite-matching algorithm is to allow *unmatching and rerouting* existing matches when a better overall assignment exists, which pure greedy assignment never does.
- Why this is the optimal approach: Kuhn's algorithm augments the matching one left-node at a time: for each new left node, it tries every preferred right node, and if that right node is already matched, it recursively attempts to *re-match* the right node's current partner to a different option before giving up — this augmenting-path search costs O(E) per left node in the worst case, for O(V*E) total, which is optimal for this simple algorithm because it's guaranteed (by Berge's theorem) to find a maximum matching, unlike greedy assignment which offers no such guarantee.
- Recognize the pattern: Model matching as finding augmenting paths: try to match each left node directly; if its preferred right node is taken, attempt to free that right node by re-matching its current partner elsewhere first — only fail to match the left node if no such augmenting path exists anywhere in the graph.
- Code walkthrough:
- Iterates over each `left` node and tries its preferred `right` nodes in order, stopping at the first unmatched one.
- `matched` maps each `right` node to the `left` node it was assigned to; once claimed, a `right` node is unavailable to others.
- Produces `{'taskA': 'dev1', 'taskB': 'dev2'}` — a valid greedy matching, though not necessarily maximum for all graphs.

**Learn more:**
- Website: [cp-algorithms: Kuhn's Algorithm for Bipartite Matching](https://cp-algorithms.com/graph/kuhn_maximum_bipartite_matching.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Bipartite+matching+as+an+optional+stretch+concept+data+structures+algorithms)

**Trade-offs:**
- The greedy matching in the sample code is O(V + E) and simple, but only produces a maximum matching by coincidence on easy inputs; Kuhn's algorithm is slower (O(V*E)) but is provably guaranteed to find a maximum matching on every input, which the greedy version cannot promise.
- Kuhn's algorithm is simple to implement but O(V*E) can be slow on large dense bipartite graphs; the Hopcroft-Karp algorithm finds maximum bipartite matching in O(E*sqrt(V)) by finding multiple augmenting paths per phase instead of one at a time, and is preferred when performance at scale matters.

**Practical software engineering use cases:**
- When to use it: Use bipartite matching (via Kuhn's or Hopcroft-Karp) for task/worker assignment, resource allocation, stable matching approximations, or any 'pair up two distinct groups optimally' problem.
- When not to use it: Don't use greedy first-choice assignment when you need a *maximum* matching guarantee — it can leave nodes unmatched that a proper augmenting-path algorithm would have successfully matched by rerouting existing assignments.


In [60]:
# Greedy matching is simple but not always optimal; use it only as intuition.
left_to_right = {"dev1": ["taskA", "taskB"], "dev2": ["taskB"]}
matched = {}
for left, choices in left_to_right.items():
    for right in choices:
        if right not in matched:
            matched[right] = left
            break
print(matched)


{'taskA': 'dev1', 'taskB': 'dev2'}


In [61]:
# Practice: Advanced Graph + DP

# Problem:
#  - Describe a graph or network-style problem that needs optimization (e.g., routing, capacity, dependencies).
# Approach:
#  - Map the problem to an appropriate graph representation (directed/undirected, weighted/unweighted).
#  - Choose between BFS/DFS/DP-on-DAG/MST/flow/matching based on constraints.
# Time Complexity:
# Space Complexity:
# Edge Cases:
#  - Disconnected components, cycles, negative weights, multiple sources/sinks, etc.


## DP on Graphs and Trees

### Study checklist
- [ ] Rooted tree DP (bottom-up aggregation: subtree sums, heights, number of paths)
- [ ] Rerooting technique to compute answers for every possible root efficiently
- [ ] DP on DAGs using topological order (e.g., longest path, counting paths)
- [ ] Bitmask DP on graphs (small N: Hamiltonian paths / traveling salesman-style problems)

### Notes
Write patterns and mistakes here - especially how you identify overlapping subproblems and define dp[state] on graph/tree structures.

### Beginner-friendly intro
Tree DP and DAG DP attach a subproblem to each node, solve children first, then combine their answers at the parent — giving each node the full picture of its subtree in one bottom-up pass.
You use these techniques when the answer for a subtree depends only on its children's answers, so processing nodes in post-order or topological order avoids all redundant recomputation.
Problems here ask for subtree aggregates like sums or heights, the best answer computed for every possible root (rerooting), the longest or highest-count path in a DAG, or optimal coverage of small graphs using bitmask states.

### Checklist item: Rooted tree DP (bottom-up aggregation: subtree sums, heights, number of paths)

**Approach:**
- Why this matters: Recomputing a subtree's aggregate (sum, height, path count) from scratch every time it's needed is wasted work when the same subtree's answer is reused by its parent, grandparent, and so on; rooted tree DP names that reusable state once and computes it bottom-up exactly once per node.
- Why this is the optimal approach: Defining `subtree_sum(node) = values[node] + sum(subtree_sum(child) for child in children)` and evaluating it bottom-up visits each node exactly once and does O(children) combine work per node, giving O(n) total time across the whole tree — optimal because any algorithm that must report an aggregate depending on every node's value cannot avoid visiting all n nodes at least once, and this recursion visits each exactly once with no recomputation.
- Recognize the pattern: Define the state (what does each node's subtree contribute), the recurrence (how a node's answer combines its children's answers), the base case (a leaf's answer), and the evaluation order (children before parents, i.e., post-order) before writing any loop or recursive call.
- Code walkthrough:
- `subtree_sum` returns the node's own value plus the recursive `subtree_sum` of every child.
- Bottoms out at leaves, which return only their own `values[node]`; each parent accumulates its children's totals on the way back up.
- Prints `14` — `D=4`, `B=2+4=6`, `C=3`, `A=5+6+3=14`.

**Learn more:**
- Website: [USACO Guide: DP on Trees - Solving For All Roots](https://usaco.guide/gold/all-roots?lang=py)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Rooted+tree+DP+%28bottom-up+aggregation+data+structures+algorithms)

**Trade-offs:**
- Because each subtree's sum is computed exactly once and directly reused by its parent's sum (via the recursive call's return value), no explicit memoization table is needed here — the call stack itself acts as the memo, unlike DP problems with overlapping subproblems that require an explicit cache.
- This computes a single aggregate (sum) per subtree; if you need multiple related aggregates simultaneously (e.g., both subtree sum and subtree max), you can return a tuple from each recursive call to compute both in the same single traversal rather than doing two separate passes.

**Practical software engineering use cases:**
- When to use it: Use rooted tree DP for subtree aggregates: subtree sums/counts, tree height, diameter (via combining two subtree heights), or counting paths within a rooted hierarchy.
- When not to use it: Don't use plain bottom-up tree DP when a node's answer must also depend on information from *outside* its subtree (its ancestors) — that requires the rerooting technique, which propagates information back down after the bottom-up pass.


In [62]:
# Rooted tree DP: subtree sums.
values = {"A": 5, "B": 2, "C": 3, "D": 4}
tree = {"A": ["B", "C"], "B": ["D"], "C": [], "D": []}
def subtree_sum(node):
    return values[node] + sum(subtree_sum(child) for child in tree[node])
print(subtree_sum("A"))


14


### Checklist item: Rerooting technique to compute answers for every possible root efficiently

**Approach:**
- Why this matters: Computing 'the answer for every possible root' by re-running a bottom-up tree DP from each of the n candidate roots costs O(n^2) total; rerooting computes it in O(n) by reusing the first DFS's results and propagating a correction as the root conceptually shifts one edge at a time.
- Why this is the optimal approach: A first DFS computes each subtree's size (or aggregate) rooted at an arbitrary starting node in O(n); a second DFS then reroots by propagating a corrected 'contribution from the rest of the tree' downward, again in O(n) — total O(n) for all n roots' answers, which is optimal because you need at least O(n) just to output n answers, and O(n^2) (naively re-running the DP per root) does asymptotically more work than necessary for the same result.
- Recognize the pattern: Define the state, recurrence, base case, and evaluation order for the initial bottom-up pass exactly as in standard tree DP; then define a second, top-down pass that combines each parent's total answer with everything *except* the current child's subtree to derive that child's answer as if it were the root.
- Code walkthrough:
- `dfs(node, parent)` sets `size[node] = 1`, then recurses into every neighbour except the one it came from to avoid treating the parent as a child.
- After each child's DFS returns, `size[node] += size[neighbor]` accumulates the full subtree count.
- Prints `[4, 1, 2, 1]` — node `0` covers all 4 nodes, node `2` covers itself and node `3`, and leaves `1` and `3` each cover only themselves.

**Learn more:**
- Website: [USACO Guide: DP on Trees - Solving For All Roots](https://usaco.guide/gold/all-roots?lang=py)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Rerooting+technique+to+compute+answers+for+every+possible+root+efficiently+data+structures+algorithms)

**Trade-offs:**
- Rerooting trades the simplicity of running one bottom-up DP per root (easy to write, O(n^2) total) for a more intricate two-pass algorithm (harder to get right, but O(n) total) — only worth the added complexity when n is large enough that the O(n^2) naive approach would actually be too slow.
- This first-pass code only computes subtree sizes as setup; the second, rerooting pass (not shown here) is problem-specific — the exact 'undo the child's contribution, add the parent's outside contribution' formula depends on what aggregate you're rerooting (sum, max distance, count), so it must be derived per problem rather than reused verbatim.

**Practical software engineering use cases:**
- When to use it: Use rerooting whenever a problem asks for an answer 'for every node as if it were the root' — sum of distances to all other nodes, maximum distance from each node, or centroid-related computations across an entire tree.
- When not to use it: Don't use rerooting if you only need the answer for one specific root — a single bottom-up DP pass from that root is simpler and sufficient; rerooting's benefit only appears when you need the answer for *every* node as root.


In [63]:
# Rerooting setup: compute subtree sizes before passing parent context downward.
tree = {0: [1, 2], 1: [0], 2: [0, 3], 3: [2]}
size = [0] * len(tree)
def dfs(node, parent):
    size[node] = 1
    for neighbor in tree[node]:
        if neighbor != parent:
            dfs(neighbor, node)
            size[node] += size[neighbor]
dfs(0, -1)
print(size)


[4, 1, 2, 1]


### Checklist item: DP on DAGs using topological order (e.g., longest path, counting paths)

**Approach:**
- Why this matters: Counting the number of distinct paths to each node in a DAG by explicit path enumeration is exponential (the number of paths can itself be exponential in the number of nodes); DP on topological order counts paths by summing contributions instead of enumerating each one individually.
- Why this is the optimal approach: Propagating `paths[neighbor] += paths[node]` forward in topological order does O(1) work per edge, giving O(V + E) total time — optimal because path *counting* (as opposed to path *enumeration*) only requires knowing, for each node, the sum of ways to reach it, and topological order guarantees every predecessor's count is finalized before it's needed, so each edge contributes exactly once with no re-processing.
- Recognize the pattern: Process nodes in topological order; for each node, propagate its already-finalized value forward to each successor by accumulation (sum) or comparison (min/max, for shortest/longest path) — the node being processed must have all of its own contributing predecessors already resolved, which topological order guarantees.
- Code walkthrough:
- Seeds `paths = {'A': 1}` — exactly one way to reach the source — then propagates counts forward in topological order.
- Each node adds its own `paths[node]` to each neighbour's running total, so a neighbour accumulates contributions from all paths leading to it.
- Prints `{'A': 1, 'B': 1, 'C': 1, 'D': 2}` — `D` is reachable by two distinct paths (`A→B→D` and `A→C→D`).

**Learn more:**
- Website: [cp-algorithms: Topological Sorting](https://cp-algorithms.com/graph/topological-sort.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=DP+on+DAGs+using+topological+order+%28e.g.+data+structures+algorithms)

**Trade-offs:**
- Counting paths via DP is O(V + E) regardless of how many actual paths exist (even if that number is exponential); explicitly enumerating and listing every path is inherently exponential in the worst case, so DP is the only tractable approach once the graph is large enough to have many paths.
- This seeds `paths = {'A': 1}` assuming a single known source; for a DAG with multiple independent sources (nodes with no incoming edges), each source must be seeded with `1` before the topological propagation begins.

**Practical software engineering use cases:**
- When to use it: Use topological-order path counting for counting distinct build orders, counting ways to reach a state in a workflow DAG, or any 'how many ways' question over an acyclic dependency structure.
- When not to use it: Don't use this technique if the graph might contain cycles — a cycle means there's no valid topological order and the DAG assumption (each node's predecessors are fully resolved before it) breaks down entirely.


In [ ]:
# Count total paths from the source to every node in a DAG.
graph = {'A': ['B', 'C'], 'B': ['D'], 'C': ['D'], 'D': []}
topo_order = ['A', 'B', 'C', 'D']
paths = {'A': 1}   # one way to reach the source
for node in topo_order:
    for neighbor in graph[node]:
        paths[neighbor] = paths.get(neighbor, 0) + paths[node]
print('path counts:', paths)
print('paths to D:', paths.get('D', 0))   # 2 (A->B->D and A->C->D)


### Checklist item: Bitmask DP on graphs (small N: Hamiltonian paths / traveling salesman-style problems)

**Approach:**
- Why this matters: Problems like the Traveling Salesman Problem or Hamiltonian path (visit every node exactly once) have no known polynomial algorithm, but for small N (roughly N <= 20), bitmask DP makes the exponential state space of 'which subset of nodes have I visited' explicit and tractable, rather than re-deriving each subset's answer from scratch via brute-force permutation.
- Why this is the optimal approach: Encoding the visited-set as an N-bit integer gives O(2^N * N) states (mask, last-node) each combined in O(N) time for transitions, for O(2^N * N^2) total time — this is optimal *relative to brute-force permutation enumeration* (O(N!)) because bitmask DP reuses partial results shared across many different orderings that visit the same subset and end at the same node, whereas permutation enumeration recomputes those shared subpaths redundantly for every distinct ordering.
- Recognize the pattern: Encode the visited set as bits in an integer mask; define `dp[(mask, last)]` as the number of ways (or best cost) to have visited exactly the nodes in `mask`, ending at node `last`, then transition by trying every unvisited next node and updating `dp[(mask | (1 << nxt), nxt)]`.
- Code walkthrough:
- Encodes visited nodes as bits: `dp[(mask, last)]` counts paths that have visited exactly the nodes in `mask` and ended at node `last`.
- Seeds with `dp[(1<<0, 0)] = 1` — one path starting at node 0 with only node 0 visited — then expands to each unvisited next node.
- Sums `dp[((1<<n)-1, last)]` over all possible last nodes to get the total number of Hamiltonian paths in the 3-node graph.

**Learn more:**
- Website: [GeeksforGeeks: Bitmasking and Dynamic Programming](https://www.geeksforgeeks.org/bitmasking-and-dynamic-programming-set-1-count-ways-to-assign-unique-cap-to-every-person/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Bitmask+DP+on+graphs+%28small+N+data+structures+algorithms)

**Trade-offs:**
- Bitmask DP's O(2^N) state space is only tractable for small N (roughly N <= 20 in practice, since 2^20 is about a million); beyond that, the exponential blow-up makes it infeasible and you need an approximation algorithm or heuristic (like nearest-neighbor or genetic algorithms) instead of an exact solution.
- This counts the number of Hamiltonian paths (any starting point, any order); for the Traveling Salesman Problem specifically, you'd instead track *minimum cost* per (mask, last) state rather than a count, and add edge weights to the transition instead of summing path counts.

**Practical software engineering use cases:**
- When to use it: Use bitmask DP for small-N combinatorial optimization over subsets: Traveling Salesman on small instances, assigning a small number of tasks to workers, or any 'which subset have I processed so far' state space.
- When not to use it: Don't use bitmask DP once N grows beyond roughly 20 — the 2^N state space becomes computationally infeasible, and you need an approximation algorithm, heuristic search, or problem-specific structure to make progress instead.


In [65]:
# Bitmask DP: visited set encoded as bits.
n = 3
dp = {(1 << 0, 0): 1}
for mask in range(1 << n):
    for last in range(n):
        ways = dp.get((mask, last), 0)
        if not ways:
            continue
        for nxt in range(n):
            if not (mask & (1 << nxt)):
                dp[(mask | (1 << nxt), nxt)] = dp.get((mask | (1 << nxt), nxt), 0) + ways
print(sum(dp.get(((1 << n) - 1, last), 0) for last in range(n)))


2


In [66]:
# Practice: DP on Graphs and Trees

# Example exercise idea:
#  - Given a rooted tree, compute for every node the sum of values in its subtree.
#  - Extend it to a rerooting DP where you compute some metric for every choice of root.
#
# Problem:
# Approach:
# Time Complexity:
# Space Complexity:
# Edge Cases:
#  - Skewed trees, single-node trees, very large depth, etc.


## Tries

### Study checklist
- [ ] Trie node structure
- [ ] Insert and search
- [ ] Prefix matching
- [ ] Common problems: Implement Trie, Word Search II, Design Add and Search Words Data Structure

### Notes
Write your understanding, patterns, mistakes, and edge cases here.


### Beginner-friendly intro
A trie is a tree where each level represents a character in a string and paths represent words or prefixes.
You use tries when you want to store and search a large set of strings by prefix very efficiently.
Problems here often involve autocomplete, prefix queries, or checking whether words share a common prefix.


### Checklist item: Trie node structure

**Approach:**
- Why this matters: A trie's entire value proposition — sharing common prefixes across many words in one structure — depends on the node design storing a `children` map and an `is_word` flag; get this structure wrong (e.g., no way to mark word endings) and you can't distinguish a stored word from a mere prefix of a longer one.
- Why this is the optimal approach: Representing children as a dict keyed by character gives O(1) average lookup for 'does this node have a child for character c', so building or walking a path of length L costs O(L) regardless of the alphabet size — optimal versus a fixed-size array of children (O(1) but wastes memory for a large or sparse alphabet) when the effective alphabet per node is small, which is the common case in real text.
- Recognize the pattern: Start from the brute-force version (a plain list of words, checked with linear scan or `in`), name the state that removes repeated work (the fact that many words share prefixes, so re-walking a shared prefix for each word wastes comparisons), then implement the smallest structure that preserves that shared state — one node per distinct prefix, not per word.
- Code walkthrough:
- Defines `TrieNode` with `children` (a dict mapping chars to child nodes) and `is_word` (a bool flag at word endpoints).
- Manually inserts `'hi'` by calling `setdefault` twice — once for `'h'` at the root and once for `'i'` at the h-node.
- Verifies the structure: prints the root's children keys (`['h']`) and the `is_word` flag at the leaf (`True`).

**Learn more:**
- Website: [GeeksforGeeks: Trie Data Structure](https://www.geeksforgeeks.org/trie-insert-and-search/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Trie+node+structure+data+structures+algorithms)

**Trade-offs:**
- A dict-based `children` map handles a large or unknown alphabet (unicode text) memory-efficiently, since only characters actually used get an entry; a fixed 26-length array (for lowercase-only text) trades that flexibility for slightly faster, cache-friendlier access at each node.
- Manually walking `setdefault` calls (as shown) demonstrates the node-linking mechanics clearly for learning, but production code should wrap this in `insert`/`search` methods (as the next item does) rather than manipulating `children` dicts directly at every call site.

**Practical software engineering use cases:**
- When to use it: Use a trie's node structure whenever you need to store and query many strings that share common prefixes: dictionaries, routing tables, or autocomplete indexes.
- When not to use it: Don't build a trie for a small, static set of strings with no meaningful prefix sharing — a set or sorted list is simpler and has less overhead for a handful of unrelated strings.


In [ ]:
# TrieNode is just a dict of children plus an end-of-word flag.
class TrieNode:
    def __init__(self):
        self.children = {}   # char -> TrieNode
        self.is_word = False

root = TrieNode()
# Manually trace inserting 'hi': root -h-> node -i-> node(is_word=True)
h_node = root.children.setdefault('h', TrieNode())
i_node = h_node.children.setdefault('i', TrieNode())
i_node.is_word = True
print('children of root:', list(root.children.keys()))
print("'hi' is a word:", root.children['h'].children['i'].is_word)


### Checklist item: Insert and search

**Approach:**
- Why this matters: Insert and search are what turn the trie's node structure into a useful data structure; without them, you just have a graph of nodes with no defined way to add or find words, so this item is where the trie starts actually answering 'is this exact word present'.
- Why this is the optimal approach: Both insert and search walk exactly one path of length L (the word's length) through the trie, doing O(1) work per character, giving O(L) time for each operation — this is optimal because you must examine every character of the word at least once to know it's present (or to insert it), and a trie achieves that with no wasted comparisons against unrelated stored words, unlike scanning a list of n words (O(n*L) worst case).
- Recognize the pattern: Start from the brute-force version (scan a list of stored words with `==` comparison, O(n*L) worst case), name the state that removes repeated work (shared prefixes across stored words), then implement insert/search that walk one path per operation instead of comparing against every stored word.
- Code walkthrough:
- `insert` walks the trie character by character using `setdefault` to create any missing node, then marks `is_word = True` at the final character.
- `search` follows the same path; returns `False` if any character is absent, and `node.is_word` at the end to distinguish full words from prefixes.
- Shows three cases: `search('app')` → `True` (inserted), `search('ap')` → `False` (prefix only), `search('apply')` → `False` (never inserted).

**Learn more:**
- Website: [GeeksforGeeks: Trie Data Structure](https://www.geeksforgeeks.org/trie-insert-and-search/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Insert+and+search+data+structures+algorithms)

**Trade-offs:**
- A trie answers exact-word search in O(L), independent of how many words are stored (n); a hash set of strings also answers exact search in O(L) average (to hash the string) — for *exact* membership alone, a trie has no complexity advantage over a set, and its real payoff only appears once prefix queries are needed (the next item).
- `search('ap')` correctly returns `False` even though `'ap'` is a valid path in the trie, because reaching a node doesn't mean that node marks a complete word — distinguishing 'this prefix exists' from 'this exact word was inserted' is exactly what the `is_word` flag is for, and forgetting to check it is a common trie bug.

**Practical software engineering use cases:**
- When to use it: Use trie insert/search for exact-word membership testing when you also anticipate needing prefix queries later — spell-checkers, dictionaries, and word-validity checks in games.
- When not to use it: Don't use a trie if you only ever need exact-match membership testing with no prefix queries — a hash set gives the same O(L) average lookup with less implementation complexity.


In [ ]:
# Insert adds a word char by char; search checks exact membership.
class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_word = False

root = TrieNode()

def insert(word):
    node = root
    for ch in word:
        node = node.children.setdefault(ch, TrieNode())
    node.is_word = True

def search(word):
    node = root
    for ch in word:
        if ch not in node.children:
            return False
        node = node.children[ch]
    return node.is_word

insert('apple')
insert('app')
print(search('app'))    # True  -- exact match
print(search('ap'))     # False -- not a complete word
print(search('apply'))  # False -- never inserted


### Checklist item: Prefix matching

**Approach:**
- Why this matters: Prefix matching is the operation a hash set fundamentally cannot do efficiently — a set can tell you if an exact string exists, but not 'does any stored word start with this prefix' without scanning every entry; this is the trie's actual unique advantage over simpler structures.
- Why this is the optimal approach: Walking to the end of the prefix costs O(P) (P = prefix length), and collecting all completions from that point costs O(K) where K is the total size of the matching subtrie — giving O(P + K) total, which is optimal because you must at least traverse the prefix path once and touch every matching result once to report it; no algorithm can list K results in less than O(K) time.
- Recognize the pattern: Start from the brute-force version (scan every stored word and check `word.startswith(prefix)`, O(n*P)), name the state that removes repeated work (all words sharing a prefix live under the same trie node), then implement prefix matching as 'walk to the prefix node once, then explore only its subtree'.
- Code walkthrough:
- `starts_with` descends to the end of the prefix and returns `True` if every character existed — the prefix node is reachable.
- `words_with_prefix` descends to the prefix node, then `collect` recursively gathers every word in that subtrie by DFS.
- For inserted words `['car', 'card', 'care', 'cat']`, `words_with_prefix('car')` returns `['car', 'card', 'care']`.

**Learn more:**
- Website: [GeeksforGeeks: Trie Data Structure](https://www.geeksforgeeks.org/trie-insert-and-search/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Prefix+matching+data+structures+algorithms)

**Trade-offs:**
- Prefix matching via a trie is O(P + K), independent of how many *other*, non-matching words are stored (n); the brute-force `startswith` scan is O(n*P) because it must check every stored word regardless of relevance — the trie's advantage grows precisely as the ratio of irrelevant-to-relevant words increases.
- `words_with_prefix` collects all completions into a list eagerly; for a very large matching subtrie (e.g., a common one-letter prefix), a generator-based `collect` that yields lazily would avoid materializing every match before the caller even needs them, at the cost of a slightly less direct return value.

**Practical software engineering use cases:**
- When to use it: Use trie-based prefix matching for autocomplete, IDE symbol lookup, command-palette fuzzy-prefix search, or any 'show me everything that starts with what the user typed so far' feature.
- When not to use it: Don't use a trie for prefix matching if prefixes are rarely or never queried — the extra memory and implementation complexity of a trie isn't justified if a hash set (with no prefix support) would suffice for your actual query patterns.


In [ ]:
# starts_with checks if any word begins with the prefix.
# words_with_prefix collects every inserted word that does.
class TrieNode:
    def __init__(self):
        self.children = {}
        self.is_word = False

root = TrieNode()

def insert(word):
    node = root
    for ch in word:
        node = node.children.setdefault(ch, TrieNode())
    node.is_word = True

def starts_with(prefix):
    node = root
    for ch in prefix:
        if ch not in node.children:
            return False
        node = node.children[ch]
    return True

def words_with_prefix(prefix):
    node = root
    for ch in prefix:
        if ch not in node.children:
            return []
        node = node.children[ch]
    results = []
    def collect(n, current):
        if n.is_word:
            results.append(current)
        for ch, child in n.children.items():
            collect(child, current + ch)
    collect(node, prefix)
    return results

for word in ['car', 'card', 'care', 'cat']:
    insert(word)
print(starts_with('car'))          # True
print(starts_with('dog'))          # False
print(words_with_prefix('car'))    # ['car', 'card', 'care']


### Checklist item: Common problems: Implement Trie, Word Search II, Design Add and Search Words Data Structure

**Approach:**
- Why this matters: This item exists to turn trie pattern-recognition into a working solution on a genuinely different problem (wildcard search) than the plain insert/search/prefix items already covered, so handling a search query with an unknown character gets real practice.
- Why this is the optimal approach: Exact search matches the trie's structure directly in O(L) as before; a wildcard character forces trying every child at that position, so worst-case time becomes O(26^W * L) where W is the number of wildcards — this is still far better than the brute-force alternative of comparing the wildcard pattern against every stored word individually (O(n*L)), because the trie prunes entire branches the instant a non-wildcard character fails to match, rather than checking full words that share no useful prefix.
- Recognize the pattern: Use the list as a practice queue: pick one problem, write the brute-force version first (compare the wildcard pattern against every stored word with a character-by-character match function), identify that most stored words share no relevant prefix with the query and can be skipped entirely, then replace the full scan with a trie walk that only branches (tries every child) at wildcard positions.
- Code walkthrough:
- Defines `WordDictionary` as a trie where each node doubles as itself (`self.children`, `self.is_word`), with `add_word` walking/creating the path exactly like a standard trie insert.
- `search` uses a recursive `dfs(node, i)`: on a literal character, it follows the single matching child (or fails immediately if absent); on `'.'`, it branches into *every* child and returns `True` if any branch's search succeeds.
- On the trie containing `['bad', 'dad', 'mad']`: `search('pad')` is `False` (no `'p'` branch exists at the root), `search('.ad')` is `True` (the wildcard matches `'b'`, `'d'`, or `'m'`), and `search('bad')` is `True` (an exact, literal match).

**Learn more:**
- Website: [GeeksforGeeks: Trie Data Structure](https://www.geeksforgeeks.org/trie-insert-and-search/)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Implement+Trie+data+structures+algorithms)

**Trade-offs:**
- Wildcard search costs O(26^W * L) in the worst case (W wildcards, each branching into up to 26 children); this is only acceptable because W is typically small in practice — a query that's mostly or entirely wildcards degrades toward scanning most of the trie, at which point a trie offers little advantage over a plain word list.
- This trie stores each word as a full character path (O(L) nodes per word in the worst case, though prefix-sharing reduces this); a hash set of words would give the same O(1)-average exact search with far less memory overhead, but cannot support wildcard queries at all — wildcard support is the entire reason to pay the trie's structural cost here.

**Practical software engineering use cases:**
- When to use it: Use a trie with wildcard search for dictionary games (word-guessing with unknown letters), fuzzy command matching, or search-with-placeholders features.
- When not to use it: Don't use trie-based wildcard search when the wildcard could appear anywhere in an unbounded number of positions on a huge dictionary — the branching factor can explode; consider a different indexing strategy (like n-gram indexes) for that scale.


In [ ]:
# Tries - Common problems: Implement Trie, Word Search II, Design Add and Search Words Data Structure
# Design Add and Search Words Data Structure solved in full below; Implement Trie is already covered earlier in this section.
class WordDictionary:
    def __init__(self):
        self.children = {}
        self.is_word = False

    def add_word(self, word):
        node = self
        for ch in word:
            node = node.children.setdefault(ch, WordDictionary())
        node.is_word = True

    def search(self, word):
        def dfs(node, i):
            if i == len(word):
                return node.is_word
            ch = word[i]
            if ch == ".":
                return any(dfs(child, i + 1) for child in node.children.values())
            if ch not in node.children:
                return False
            return dfs(node.children[ch], i + 1)
        return dfs(self, 0)

wd = WordDictionary()
for w in ["bad", "dad", "mad"]:
    wd.add_word(w)
print(wd.search("pad"))   # False -- no branch starts with 'p'
print(wd.search(".ad"))   # True  -- wildcard matches 'b', 'd', or 'm'
print(wd.search("bad"))   # True  -- exact match

practice_queue = [
    {'problem': 'Word Search II', 'topic': 'Tries', 'status': 'todo'}
]

for entry in practice_queue:
    print(f"{entry['topic']}: {entry['problem']} -> {entry['status']}")


In [71]:
# Practice: Tries

# Problem:
# Approach:
# Time Complexity:
# Space Complexity:
# Edge Cases:



## Disjoint Set Union / Union Find

### Study checklist
- [ ] Parent array
- [ ] Find with path compression
- [ ] Union by rank / size
- [ ] Connected components
- [ ] Common problems: Redundant Connection, Number of Connected Components, Accounts Merge

### Notes
Write your understanding, patterns, mistakes, and edge cases here.


### Beginner-friendly intro
Disjoint Set Union (Union-Find) is a structure that keeps track of which elements belong to the same group or component.
It supports two operations: find (which group an element is in) and union (merge two groups).
You use it when you repeatedly connect items and need to know whether they are already connected.


### Checklist item: Parent array

**Approach:**
- Why this matters: A parent array is the minimal representation needed to answer 'which group does this node belong to' — starting with every node as its own root (its own group) is the baseline every union and find operation builds on top of.
- Why this is the optimal approach: Initializing `parent[i] = i` for all i is O(n), which is unavoidable since you must create a representation for every node; the naive `find_naive` that walks parent pointers without compression is O(depth) per call, worst case O(n) for a degenerate chain — this naive version is intentionally not yet optimal, since it's the baseline the next two items (path compression, union by size) improve upon.
- Recognize the pattern: Represent each component by a parent pointer, starting with every node as its own root; without path compression or union by rank yet, chains can grow arbitrarily long, which is exactly the problem the next two items in this section solve.
- Code walkthrough:
- Initialises `parent = list(range(6))` so every node is its own root (`parent[i] == i`).
- `find_naive` walks `parent[x]` upward until `parent[x] == x`, without any path shortcutting.
- After manually setting `parent[1] = 0` and `parent[2] = 1`, `find_naive(2)` walks `2→1→0` and returns `0`.

**Learn more:**
- Website: [cp-algorithms: Disjoint Set Union](https://cp-algorithms.com/data_structures/disjoint_set_union.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Parent+array+data+structures+algorithms)

**Trade-offs:**
- This naive parent array with no compression can degrade to O(n) per find if unions happen to build a long chain (e.g., always attaching the new node under the previous root); this is the deliberate baseline this section improves on with path compression and union by size next.
- Manually setting `parent[1] = 0` and `parent[2] = 1` demonstrates the mechanics directly, but production code should always go through a `union(a, b)` function rather than manipulating the parent array by hand, since a bare assignment doesn't check the union-by-size invariant.

**Practical software engineering use cases:**
- When to use it: Use a plain parent array as the conceptual starting point when learning Union-Find, or when component sizes are guaranteed small enough that O(depth) finds are acceptable without further optimization.
- When not to use it: Don't ship a parent array without path compression or union by size to production — an adversarial or unlucky union order can degrade every subsequent find to O(n), erasing the whole point of using Union-Find.


In [ ]:
# Parent array: each node starts as its own root.
n = 6
parent = list(range(n))  # parent[i] == i means i is a root
print('initial parent array:', parent)

# Naive find: walk up until reaching a root (no path compression yet).
def find_naive(x):
    while parent[x] != x:
        x = parent[x]
    return x

# Manual union: point one root to the other.
parent[1] = 0
parent[2] = 1
print('find(2):', find_naive(2))  # 0 -- walks 2->1->0
print('parent after manual unions:', parent)


### Checklist item: Find with path compression

**Approach:**
- Why this matters: Without path compression, repeatedly calling find on a long chain re-walks the same pointers every single time, doing the identical O(depth) traversal over and over; path compression fixes this by permanently shortcutting the path the very first time it's walked.
- Why this is the optimal approach: Setting `parent[x] = find(parent[x])` during the recursive unwind makes every node on the path point directly to the root once `find` returns, so future finds on any of those nodes become O(1) — this is optimal because it converts O(depth) one-time work into a permanent O(1) shortcut, and amortized across many calls (especially combined with union by size), it's what drives Union-Find's near-constant O(alpha(n)) amortized time per operation.
- Recognize the pattern: Represent each component by a parent pointer, and compress the path during every find call: as the recursive `find` unwinds, overwrite each visited node's parent to point directly at the discovered root instead of its immediate former parent.
- Code walkthrough:
- Builds the chain `2→1→0` in `parent`, then calls `find(2)` which compresses the path on the way back up.
- The recursive `find` sets `parent[x] = find(parent[x])`, so after the call `parent[2]` points directly to `0`.
- The before/after prints confirm the flattening: `parent[2]` changes from `1` to `0`.

**Learn more:**
- Website: [cp-algorithms: Disjoint Set Union](https://cp-algorithms.com/data_structures/disjoint_set_union.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Find+with+path+compression+data+structures+algorithms)

**Trade-offs:**
- Path compression is pure upside for correctness and speed with essentially no downside except a marginally more complex `find` function (recursive, or iterative with a two-pass approach); there's no scenario in this section where you'd deliberately skip it.
- The recursive `find(x): parent[x] = find(parent[x])` implementation is elegant but uses O(depth) call-stack space per call before compression kicks in; for extremely long uncompressed chains (rare once compression has run a few times), an iterative two-pass version avoids the recursion depth risk entirely.

**Practical software engineering use cases:**
- When to use it: Always use path compression in any real Union-Find implementation — it costs nothing extra to add and is one of the two optimizations (with union by size) needed to reach near-constant amortized time.
- When not to use it: There's no real reason to skip path compression in production code; it's only omitted in the earlier item for pedagogical purposes, to show the difference it makes.


In [ ]:
# Path compression makes every node on the find path point directly to the root.
parent = list(range(6))

def find(x):
    if parent[x] != x:
        parent[x] = find(parent[x])  # compress path on the way back up
    return parent[x]

# Build a chain: 2 -> 1 -> 0
parent[1] = 0
parent[2] = 1
print('before find(2), parent:', parent)
find(2)   # compresses: parent[2] now points directly to 0
print('after  find(2), parent:', parent)


### Checklist item: Union by rank / size

**Approach:**
- Why this matters: Path compression alone still allows a union to attach a large tree under a small one's root, temporarily creating a taller structure than necessary before the next find flattens it; union by size prevents that from happening in the first place by always attaching the smaller tree under the larger.
- Why this is the optimal approach: Comparing `size[ra]` and `size[rb]` and always attaching the smaller-sized root under the larger's is O(1) extra work per union, and it guarantees that any node's distance to its root at least doubles the component size each time that node's root changes — meaning a node's root can change at most O(log n) times ever, which combined with path compression gives the well-known amortized O(alpha(n)) bound, the best known for this problem.
- Recognize the pattern: Represent each component by a parent pointer, compress paths during find, and — critically — always attach the smaller tree under the larger during union, tracking each root's size so this comparison is O(1).
- Code walkthrough:
- `union(a, b)` finds both roots and attaches the smaller-sized root under the larger, keeping the tree shallow.
- After `union(0,1)` and `union(0,2)`, node `0` becomes the root of all three; `size[0]` grows to `3`.
- Verifies with `find(0) == find(1) == find(2)`, which evaluates to `True`.

**Learn more:**
- Website: [cp-algorithms: Disjoint Set Union](https://cp-algorithms.com/data_structures/disjoint_set_union.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Union+by+rank+%2F+size+data+structures+algorithms)

**Trade-offs:**
- Union by size (tracking component sizes) and union by rank (tracking approximate tree height) both achieve the same asymptotic bound when paired with path compression; size is often preferred because it's simpler to reason about and is directly useful data (component sizes) that many problems need anyway.
- This maintains `size` as an array parallel to `parent`, adding O(n) space that's typically negligible compared to the parent array itself; the alternative of skipping size tracking (arbitrary attachment direction) risks the tree height growing unboundedly, which path compression alone mitigates but doesn't fully prevent as cheaply.

**Practical software engineering use cases:**
- When to use it: Always combine union by size (or rank) with path compression in any Union-Find implementation you intend to use beyond a toy example — the two optimizations together are what deliver the near-constant time guarantee.
- When not to use it: There's no real reason to skip union by size once you're implementing union at all; the only omission worth making deliberately is in a teaching context, exactly as the earlier 'parent array' item does.


In [ ]:
# Union by size prevents tall trees: always attach the smaller root under the larger.
parent = list(range(5))
size = [1] * 5

def find(x):
    if parent[x] != x:
        parent[x] = find(parent[x])
    return parent[x]

def union(a, b):
    ra, rb = find(a), find(b)
    if ra == rb:
        return False          # already in the same component
    if size[ra] < size[rb]:   # attach smaller tree under larger
        ra, rb = rb, ra
    parent[rb] = ra
    size[ra] += size[rb]
    return True

union(0, 1)
union(0, 2)
print('sizes after unions:', size)
print('all three in same set:', find(0) == find(1) == find(2))


### Checklist item: Connected components

**Approach:**
- Why this matters: Counting connected components via Union-Find (union every edge, then count distinct roots) answers the same question as graph-traversal-based component counting, but does so with a fundamentally different mechanism — incremental merging rather than exploration — which matters when edges arrive one at a time (as in a stream) rather than as a complete graph upfront.
- Why this is the optimal approach: Unioning all E edges costs O(E * alpha(n)) with path compression and union by size, and counting distinct roots afterward costs O(n) (one find per node), for O((n + E) * alpha(n)) total — effectively linear, and optimal because you must process every edge and every node at least once to know the final grouping, with alpha(n) being the near-constant overhead of the two DSU optimizations.
- Recognize the pattern: Union all edges using find-with-path-compression and union-by-size, then count distinct roots by checking how many nodes satisfy `find(i) == i` — each such node is the representative of exactly one component.
- Code walkthrough:
- Calls `union(a, b)` for every edge, merging each pair into a single component.
- After processing `[(0,1), (2,3)]`, nodes `0,1` share one root, `2,3` share another, and `4` is its own root.
- Counts distinct roots with `len({find(i) for i in range(n)})` and prints `3`.

**Learn more:**
- Website: [cp-algorithms: Disjoint Set Union](https://cp-algorithms.com/data_structures/disjoint_set_union.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Connected+components+data+structures+algorithms)

**Trade-offs:**
- Union-Find counts components incrementally as edges are unioned, making it well-suited to streaming or online settings where edges arrive over time and you need the current component count at any point; a DFS/BFS-based approach requires the full edge list upfront and would need to be rerun entirely if new edges arrive later.
- This counts components in O((n + E) * alpha(n)); a DFS/BFS-based approach is O(n + E) exactly, technically simpler for a one-shot count on a fully-known static graph — Union-Find's advantage appears specifically when edges are added incrementally, not for a single static count.

**Practical software engineering use cases:**
- When to use it: Use Union-Find for connected-component counting when edges arrive incrementally (streaming graph updates, online network monitoring) or when you also need fast 'are these two nodes connected' queries interleaved with the counting.
- When not to use it: Don't use Union-Find for a one-time component count on a graph you already have fully in memory — a single DFS/BFS pass is simpler to write and equally linear-time for that static case.


In [ ]:
# Count connected components using Union-Find (edge-list input).
edges = [(0, 1), (2, 3)]  # 0-1 connected, 2-3 connected, 4 isolated
n = 5
parent = list(range(n))
size = [1] * n

def find(x):
    if parent[x] != x:
        parent[x] = find(parent[x])
    return parent[x]

def union(a, b):
    ra, rb = find(a), find(b)
    if ra == rb:
        return
    if size[ra] < size[rb]:
        ra, rb = rb, ra
    parent[rb] = ra
    size[ra] += size[rb]

for a, b in edges:
    union(a, b)

components = len({find(i) for i in range(n)})
print('components:', components)  # 3


### Checklist item: Common problems: Redundant Connection, Number of Connected Components, Accounts Merge

**Approach:**
- Why this matters: This item exists to turn Union-Find pattern-recognition into a working solution on a genuinely different problem (cycle detection while building a graph incrementally) than the parent-array, compression, and component-counting items already covered, so 'detect the exact edge that creates a redundant connection' gets real practice.
- Why this is the optimal approach: Processing edges one at a time and unioning each pair costs O(alpha(n)) amortized per edge with path compression and union by size, for O(E * alpha(n)) total across all E edges — this is optimal because the answer (the first edge whose endpoints are already connected) requires processing edges in order and checking connectivity incrementally, which is exactly what Union-Find does in near-constant time per edge, versus re-running a full graph traversal after adding each edge (O(E * (V + E)) in the worst case).
- Recognize the pattern: Use the list as a practice queue: pick one problem, write the brute-force version first (after adding each edge, run a full traversal to check whether a cycle now exists), identify that Union-Find already tracks connectivity incrementally without a fresh traversal, then replace the per-edge traversal with a single `find`-and-compare check per edge.
- Code walkthrough:
- Defines `find_redundant_connection(edges)`: processes edges in the given order, and for each edge `(a, b)`, checks whether `a` and `b` are already in the same component via `find`.
- If `find(a) == find(b)`, the edge connects two nodes already reachable from each other — adding it would create a cycle, so this edge is returned immediately as the redundant one.
- Otherwise, the edge is a genuine new connection, so the two components are merged with `parent[ra] = rb` before moving to the next edge.
- On `[[1,2], [1,3], [2,3]]`, the first two edges connect 1-2 and 1-3 (building one component `{1,2,3}`); the third edge `(2,3)` connects two nodes already in that component, so it's returned as the redundant connection.

**Learn more:**
- Website: [cp-algorithms: Disjoint Set Union](https://cp-algorithms.com/data_structures/disjoint_set_union.html)
- YouTube: [Search this topic on YouTube](https://www.youtube.com/results?search_query=Redundant+Connection+data+structures+algorithms)

**Trade-offs:**
- This returns the *first* edge (in input order) that would create a cycle, which matches the standard problem definition (the tree was valid until this specific edge was added); if instead you needed *all* redundant edges (not just the first), you'd continue processing and collect every edge that fails the union check rather than returning immediately.
- Using Union-Find here is O(E * alpha(n)); an alternative that builds an adjacency list and DFS-checks for a cycle after adding each edge is O(E * (V + E)) in the worst case — Union-Find is the clear choice whenever edges are processed incrementally and you need to know 'does this specific edge create a cycle' rather than analyzing the whole graph after every addition.

**Practical software engineering use cases:**
- When to use it: Use incremental Union-Find cycle detection for validating that a graph stays a tree/forest as edges are added, network redundancy detection, or account-merging problems where each new link might already be implied by existing links.
- When not to use it: Don't use this incremental approach if all edges are known upfront and cycles must be found using edge weights (like detecting the max-weight cycle) — that requires a different algorithm than simple connectivity checking.


In [ ]:
# Disjoint Set Union / Union Find - Common problems: Redundant Connection, Number of Connected Components, Accounts Merge
# Redundant Connection solved in full below; the remaining problems stay on the practice queue.
def find_redundant_connection(edges):
    n = len(edges)
    parent = list(range(n + 1))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for a, b in edges:
        ra, rb = find(a), find(b)
        if ra == rb:
            return [a, b]
        parent[ra] = rb
    return []

print(find_redundant_connection([[1, 2], [1, 3], [2, 3]]))  # [2, 3]

practice_queue = [
    {'problem': 'Number of Connected Components', 'topic': 'Disjoint Set Union / Union Find', 'status': 'todo'},
    {'problem': 'Accounts Merge', 'topic': 'Disjoint Set Union / Union Find', 'status': 'todo'}
]

for entry in practice_queue:
    print(f"{entry['topic']}: {entry['problem']} -> {entry['status']}")


In [77]:
# Practice: Disjoint Set Union / Union Find

# Problem:
# Approach:
# Time Complexity:
# Space Complexity:
# Edge Cases:

